# CIC-DDoS2019 LightGBM CPU baseline

Smoke: 2,000 rows per source file and at most 10 new iterations in this session; target remains exactly 100. Resume/checkpoint state is synchronized with S3.


In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import os
import subprocess
import sys
import time
import zlib

PROJECT_NAME = "Luan-Van-LightGBM-Parquet-Github-v2"
SESSION_MAXIMUM_HOURS = 12.0
SESSION_STOP_BEFORE_MINUTES = 30.0
os.environ["PIPELINE_SESSION_DEADLINE_EPOCH"] = str(
    time.time() + SESSION_MAXIMUM_HOURS * 3600.0 - SESSION_STOP_BEFORE_MINUTES * 60.0
)
os.environ.setdefault("MALLOC_ARENA_MAX", "2")
PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME
SOURCE_DIR = PROJECT_DIR / "source"
PREPARED_DIR = PROJECT_DIR / "prepared"
RUNS_DIR = PROJECT_DIR / "runs"
for directory in (SOURCE_DIR, PREPARED_DIR, RUNS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

encoded_files = json.loads("{\"checkpoint.py\": \"eNrtPWtv3EaS3wPkP3B5EEJmR7QdJ4vF5OZwsWMnweZh2M7uLXQCQQ17JK445BwfthWt/vtVVT/YL3I4Izl3B5yBxEOyu7q63lX9cBiGz5usvTptsw0Lfiwur7rvnv0UPKvrtmNNsL5i6+tdXVRdG2RVHrxjTbEpWB5kXb0t1sGbp0F7U62TMAw//eTTTzZNvQ3SdNN3fcPSNCi2u7rpoGdVd1lX1FWLrcTbdftO/b4CFMriQj3/o60r9VDWl5dFdame61b9bK/6rijVY1dsmXro+yIXKOVZx/CbREg+L6jHb3XFRMNd1iEest0reBRfupsd4CA/fFPdLIKfsh2+WwRv2H/1rFoznNynn/z4y3ffvXgdrCTeySXrfoSfrInStMq2QJiYt8zZJui7dVrV76M4OP23oO2a5aefBPCnYUDCSiGaYBOJawJ94qRo603dbLMu0qC1V9kXX/0p3RQli3AuS5rCAvjYV9fpxU3H2mUA3ATsnjz+4svgc/rLGjsvLlmLTQRXEg4Vx8HP74vuigiV1DtWRWFzEcZB1kLrKi+ZgEENrwCN4KKs19fBciW+Jw3L8kjDJ9Z6DKMn/Q7nHlHv2KQJb3DFPvBf+vzXWVVXxTorU8QdSHBT1lm+RH5ZkwR+1TkI8opkLcn77a6V7RfwtUUJztp1UaxeZmULotIC49NrdtOu3jY9PrNd1oAeNO0qChfhIgiXYRwnHHAU9t3m9M+hibpFUIFD7J8LV7EU0UsRPWM2iyCH5kVFWsW5TBP8GeRDsnFokACmrOqS7XVeNBF/kNNgH4q2S+trehTodgzFPGtugDw6GGR92vabTfEh0t/zV8EfgzDptrtQFxQFSkjL+3DBaQ+asZJE8oqP4svAFt7Gz56iymFWqy9sRsUDRCGB75sCJCv8zyp0v23KHuRGe1+3yQZNXCQbgExXdRSLJvC5YbsyW7NIzdTgjcvQK6B33dxwnoqHpbIhZ8KqnIGoLpDT5+cLIkVq6HP7Tnu2Oe9ITgmjyKFiDZyYhIR2HzFRMHQZUS/9AgKepMzRIoJRQhRxzucA6+xcfK8b0Jx13eTA3UCSauANfgc+40fezDIlxYY+g/PBJtpoZjMTlQToz6o8gp6HyvEiqNj7sqjYKhwziSh6DSdX8m2x7v5GLyIp2AMaq+FnbPfnEnwFhhS6jnxt6vetYvlHEnPJXk3G32VlgYZbSvlMAV/3DQpbiogLkwZeypZr9bVFIYEGEWf7Wai+hOexX2yESLEPO7buyO6TUjRZdcmiJ2g+usjBIgaJfSJpANKkIfCHlYKl8bfJipYFf83Knr1omrqJTEnbhAKbBFUw2PbgZtd11WWAJ/5dVH3dt0FfFUArfbQnSXLrYHf3dRDa8OuLljXvYH4oHavbAcTZ8qvzuwBGKo23p18tz+80KIKV6zJrW4ju3gCykvoQ40G4l+XZDkW4bzEcGixAv+MmmkeIawIPfK13ICdD5Mjfy3CRe6kNRIxFVXRpGrWs3GCnalNcLgNHUFDjsouS5WkN4JoiZ8vgoq7L4J8kJcBT/MuWGrJwBBJ8BjIeu0T8zVkoIILcDK0RjUR8QF0dOoMM2CgERctHZ+CGOGy7iQ26AVUoPLhwGqbys4vTNvsAX7umYKgB8KQkV0DQGkBvd9wO/M5F1rK0Bf2ocoSygTEHAG4TF4uLfn3NMD4Eq8Cqd0UDHIUYNwI+KTi8TQqfoT+ERmGcwOdiF9nAdg1DzzAJjLfxARN/h490V96wSxCyaZC8zTR+wGx0Hbyp5TPmjLHJyvIiW1+ncwYTDBJQxQ8wZD9TcmKTrC0uKzA8EAuvu0E/TD0YeqkOZK1dpEGv01evX7z54bufX3ybPv/l55c/fJe++ubt9+E4ZUyYFnXIY2JkEpnNYu4858V//jnLgB01RTir2OwzqGuqJBXZYsKRImqIt9Vdyaave/tUfAcIXjEUhNJVBg2h/vyHlYuthwKOVwlfKXLI/mRYAwE3LzYb1rQBZa7A3Ge/Pv/Li7djyIlpKuTEs4kcf3kv5ARcGzmQvJc//IeNnGlpHCp5GituOVhb+pOuywIcKeVRYyojaSO9ABInQmug4wXqqV7xkexkFqK3NrtE8KEolVyB0hW/cYIIQ98OLKJxBpr4+SX9wygzXvcV1go4OwQKFnl5kSJ5nzUVKGIUnrRfB5BqZ6VR8mnYFkMT6SMXgR+Y5S8pIZP+/d93Deh8090M7p6TnxSKfDWwYelSXnBJuldrsp2RCKi+vDpzUXf1U/Mr+7Bmuy74gRoQYdDmwNt5RAwJJOKifPf7K1YFHqZCG0mvmMs4jOKhl5zfiqObCKqAXQE6C4+B4f9K8wyGk6OKgg5KD6mA6v+A4BTTYBFVNawEBN9ByATGyiqGcID8O3kf/jMp/e6VRt6Et5rg3z26lb3uQtuyUGQkP+tYNn2lYwhPRU74zUF3mL8+WUCLw9EwkvP47NFn8V0YG6EnBTwCARRUkX0QDmvhwIf6kSmqECRDiIMCsgxekIChAIxYFMxKsg4DZspGVeLhBHaYcsySdkECiWUUe0V+QGxU4Id5BCtXWoVSStz/1cHYA5GTt8ywMjAWen4efBF8/nkQScCnMG8/JNtYgdadtAqhk/zRSR5sILMBrYxO2vjrgEajcm0VnCRPNm2oMXche7q0X2CZl0VAgjiRtdoFn8gIalScbUvGdpHdDDIohtXrgbhgGtBfmHLBrc3QypJOSATbrh0zlq5gCBMoOw4f51pAn/WT0AwDiAI9hGXAkw5kukXX7jV8QloHxLRpDlFiXr+vKLTjGgkaPRgAIx+kUtE/bdcgaG6FjF66v8OIRYqnasvjYYkFys3tXUwvh3qQLAZABtx2WbVmEcEinGInT6BZ34YAIlwGomGI2bl8vHNoRK9HCLTLmi7liaKgUVv3zZrJyuCmqLIyNeimpcgadpCD/0pgIOYAqdn2HTot5CiY0zX4edQfHC3owL31l1dBJiK407LYFlhDefXLm7c8lz+C/lSlyd6LEA1TexBmVFgfQ7RZS54ABts2sulNDY24fRgjTpqJOF0RDgNf8CIaJHAlnMgJmoS70GPwRPWGzbHblB0JgHuWUEx7Cm69BjOBmEUeY2D/EaW8lrHr6HE83s4IJZTFiRMcLBrvxldRYIpnYd+U4fliummeddkK89VIdKICJ5YX9nTEUmS7ug3xb9CZSGOFXBGI7/bAQFtW993qyZ8fP55oqkIDl0Ttrq5aZS9E2BBqeik5A+Kpfsc+p80hJWRnU9CCFIxI17dRPOrhseR+rG+fkTQoq+9+sjyvv6+W8IEXIJPBVVU4ZZlZ0BeeUqD3Bh++hP/CEX5wNo989Lhpt+E4QUWGYr3l9kmkLciUYr1l3VWda4YY7BjwrK9yHF2L+pShNUJUJTRgxCDoaLATWH/5gVsyj4sUIkKm77V4+glAoA5pLin8/u3bV29Iep7XOQMbsloFXz7+EhNTNH22K9LBkmfXYREEqvdo1WC0x7chgMQvP9dv+vXVX9gNf+heIh3CO91R1RcUiNNiUevz4RaJRIlNTyD9DtTimJZguD7GgsAdi1RcT6hhenYNLeq5nCtFE3bCBaFZDooFFr6ofHuRZ0vXLmMHaUSp8/liMHBfPLbtqTs3JQnc8qS4Ci1E56jZ7rVlXjvm0tYwrTizlMsTyJtBDJ7uJlqL6BlVZFZadWYRgKSukLdzkJlnUFVtwjAD8WFE45Z20BlR9eeLtofEdA+gR23xG8oqLiEIt4oMpLp6it800lFdqO23KNraLhPeLX4A1bRCPjPaw1eKDIdpKwVyALsvO48VMOJp7nKCsYFk6VsDOD43i/BaJ7fdUCbcgGHVsiouFKfCnaoUloLnW4XmXXg/v4+aMlUs9EQC4w2nzIXIIleOTXMYGu560DCcYBhQBdLPW//bewTZKtCWw88JtA8Jtj2pcAKDSXs+TBuMOoXMcmuAHsKOxqkzY1U1yGKYaDwd+c6KV3m4jlKJtnmvExxxhhK38Q7SGygK2i5xoOGEY5yIE82JzJ47VgUqaaioJ9+k0coQq+oAydMfWXXZXQHxT5/EMaabaGon5Isr6Q+/OOsrNBAVodGO60vsIvQORxDds8XPT4pL1h3DUuh2BEeRYIKhA4BzqrGwbCs2JRk6cQx3AfR8wQabyzcz0mYi2R03U6RrzlmxrxG5sdL3V07wFkSGgO4xL+O7Ij0A3R2StJgnHPi9BO3N99+cgqAcImujSexhiewBTm0ioZ2d1LqJrTenBa9ebLL1AXnt3tx2dn47Id3jTvhfgmfZ+hrmjquz2x0w8KIoi+6Gr0+UuDv35rRo255XeLdZdwrmBcuBtLyMvqhNPnaVC3yS2ln0ESpdY45XjXmc852d/HmMpDn8hGQYDecLxUFuXMGf78ZtF753LvtduMGNPW587yRmzVu4b6vnvV34qFUdds8d6MJxW116kLhp/KHOkw7Z4Q3qybhICqZpcIFhgmor+shpRw3Ee57WDXz90+PHI7I8j9sGRWYx+8A49aAY9eHi00nEDxHqh4xJHzYePSAWPSwOPSQG/Tjx59QU5gnpg8ecM+LNebHmsXHmg8eY/9sWSe4ZNcrg0BMDHlRtIXKWhxVNlPTnrGTAe6EA/GncIe8P65fzNnY8r/syp4IaH1KrhA2+kld8STWIXr5q3eDCcUGX1nOHahkeQTm9xeOBCf7vSyqpf9CLaC6F/DSbYYNEdZcWL/fYHr22rXWb8Ly0L5UXYBeBUfs2CDARTr740DXZN80lLrCqFaZlcBtyIww/pTrfja2xHhWPzTPgdih59DKAQY54Fr7eYPBM+kvhLs+Vq8Q1NytwJP/qW7YTtD3cWm7Ct3rI6DGQvDJsTPfOZzRnEJ9CKrkAM19ytW4Tkuth1Hhj5KBS34l2z2HoN6QOIM18hHBp6kWIy5dLUz2mNg9I9n0LbmGNewlX4fNfXv09vJ8uHBp76hHn0TowtaoxJ26cln0tvvwYcv9SBgejMq+thvxfCBL+p0IDf1gw4d6EhIkY4B52dkZMsDceMDJmdO1EFnvUYW1VrvGLtUpjP4LnCPVC7W4UB8vEZvYH3rqg+pnbJRD5pWexfs9u/5cwNbkbQ6rLa7lJE8gk4iUclJasYfJPl48e3WrMu3t061Mc/wTuca78nmfLJTd1PD07YA9YgD5yf8iMVef93BtZgrVWgCUWwa+vf+SGbszETacDBxcoQ5cSs8IAfaeKmUQ/RPp8UInHf3B7ThXayLvVmL/PUo9xS8FY9j3n3oHZPtBYJjZM5ki44ZpV5QUwQqO8RKEV2xtwDsXdi7ffjwGJB34Ls+pjydCor8qiuo4Mr4GLIHR7gm8Hm/fI5bRfMA6K7bX5xm7E32nDm9i3LfZg+7ZrH7u7TdHyoC1uruGYv8Ht0F1pqj1xPPbu3zQ0BOuBkxvThgZz96UJVNTJ1jaSg5+Fz+r8Jjzn9+bEEIcZ18sY5/Wfq6N7P2VVdsnUhT76GXvzMKRGSqAuLi41dd2RtINoU0w0tNjCyCUFnDyAsjaH0cjp+Ol9beSnc5vNOfAvevjO/XONGyYGPehYsvbKOWOvJolH7NWD1cqZMEYzuLnd+WDDb59CU3GzQqQIsfBN1u6aUQaaEnS6RcSZ3qMgFI2aviKBDk3D1tYl/5gWuToYSGrHcsl1+y4F8+CdCGp4jxGFEp91+d7WnVKk9mlimlh8ox/icyYRx8oCQlNpVulcoWMnxVh4kpb/FMkn2YBQbGnGYy5ZA4KHe5lhmEr83GV9SyddQ1oJJ+sBTeumC+/8s6UzJjTOWcjpahwpl2ibvBvzTXzR3NP+iHPzPJ/ce1weEKSWH4lKFqVoKC+hrGOlg+bdpbfGNWh02n6Dj9FnJ38/2Z7kpyffn/yERzvts6WYlThnS0mmX5mXFuhHYEx94h11wC1Yd3wrnIrn7Cr/Ig+Rzrrahpb2lV7fmjSU5FqKYSz/J5m2FAPbnx0MwuXIdTd2T74EhHuToYu6qU5rpZ1dc68r49NZ+OTZoyCDXvt2hcNXd5+0BXQRzLQl3goB7s3dMuIqGxEa+3wHlzB1xFUIHO+l224EKa/ekJ3AUmsn7UN87iBgQQ3jOEjrPU6n4cYGeOeevCAzKw5YC7kxxnzkGy+OR6jvCfmPgr/QyDGUW+wiwWALVeOjrg7B3vtN4D3mSOd2yVYl3YcOpzfBYKvx9PQPRGqLJ5jX7SP9gikLHdGEUDGazUAkpbrrjcoh2kgAFvyck7IpN6BYKr3gvlNXY4PLqh2B9MSSPivLHct+bbSYpV2hJu7vc0GMEtjUYu30xIAML6bjPM5C/lZUy8+957PV+vm3giZgDkyM1TJ6XjOe0G2zbn0VeLVyFEF9sjqK8v3BSBoXoB2G4iAQpA9TlroDZ8HOzFR9we8YNF+eezL4392uU23A1Yo5dYF7GskZBpJGiOZI+cG3O4ne0+Mfan7GbiDkF6UJ8XVDonNPOk6NFxJNIwbMIKIY+DyeVOtxod6I9tN6vlzwi5f5Ha+OzVl6RVhrhzfSblu6/NaGvGEZ3crcAtrbzNukyxosXZiR6sIUeBnXajm3X9noXjndLkDvht+8SKH8jjUbSH57jNyMqstM7dMSbd58RAM1y88F120uJXoM+EHrG45oifN1grOJ8z2KZ0mvG637truQ+Q9WNnUcf0bXsDpil5BcU6PIqOHydxBOVP12wGA1hZNZfBcjy43gf5ywCnuuIuV1QzmXGVMNjVtOncozb7jQYMaurydvtTIkaNzHy07r9t14H/gYOsmT72rghYHEQodu+yJP5shdA1jPlud9T+wUj8cauKAMX8MSL4C/vNgm4gL40NucWlqxrNWu3oGSF78BhKVeoxvwAq7lfTn6fV/GqwwobfCQD3MyX+ed3cm2giJbtl87ybJme6GL9mQ39Fhi6OB56+eV2pmlx2ma9Nq9rFBt6Y3vULbikfGAjfyqdkEJjfXWaeEDCxNobeYVF4xQy1X46QjLqUwIvz4A9ZQjnDquqI+hgVS35aQNLqgTJXEfAnkC5z4mebOOB/GNIDEH497wu3z8NL+zbBiRg/7ZA9pqZXF/MQxonFM5ssqiAz4qH47njGIauANT3INGAKN5wABoo+fB12sa96i92FHvDvoaQhaZIjQasnoirOBURmDm5QN6tJFmG1riHg1s3Tonsy+hXUwHq+NrNXMDP8snH5Z/m755T9+P4aPd1Nm6UMGwyE5nZTKxmzKaD52THmUxDzQt/6/0nmo0uRe+k1FE4a7WGSmiP2fzLXGI/ZxTaQjWd7yhjre6o1/5y7dn8oh8m90EdVXeBBeM5oO3WATdFdPvUecjDDfc6xUobSPIiC3wCPQm1Kgm/ak9D9uVPsw/czInf3IyJ4fE00nK2PaYI3VPAzYuuhtDdm91QvBr8DwVE62RIdaQzqVyZ6t3xcy472YN2nFJHkTW89w9NketGYnts+MzvpVD21f+mWrqccx8UjltEyfUff8wC/65ZmwnLu9/LGpS3njzLKS10hTbG3GmUamiV6gmNXr2SI2fXJb1RWSEl587ERnucqvLnJ8tAzhny1Mc7Zz/wymAJi2w0yc7Ry9zbefUfwMwZq/q\", \"config/data.json\": \"eNqNVMlu2zAQvQfIPxhCD07gxkuWLpccGhQo0EOA9Ja4xJgcyYQpUiEpO27cf++QEhXZcIAaPow4Cx/nvZnX05PBIBPgwaHPvg5ew3d7woS0dJSNV1AUCsdSV7Ufc8mFMG42mX75WIF9rilv1CTlUiGrwHu0OiSen4/PL7qYtrIHW6Bn3Ki6DFG6Vuqoj3HQQhIQdBT2mP2EBapsNMhUMr4pcC4YPBlNgWD9aqx5Kh2TWG5NyQJMDSUymbNSOid1QRd4W2MKdlBW9BQp3oHS+cNND/Fj8OMufHxXZtMzyeoQFNbU1fGCTcSAKqf8lBXOHkxtOV1wH6reofNSg5dGtyet+95YfxiQzu6t8YZuzuZN1fn+Qx2r0Ma2JD726PDGg2LWbALW6aT5jXoFqIHMIQpyzyazG/L8jRdkjnx9VXkLMnA+ufg0SRDWoEIjCG50TK87MdBDDo56l3RHVNNjsQ16g9ob1rQZNmCx05zF51paZKAUi1qhJyPwJUsAA/mnb7gri5U1HJM2En58oXguPRPWJCojgV1DoyOKq52DPX6z4e/dI3t6cvOznIgeNvatFGe3H7LRkaihs3znIsFnKXooq11FxJ69mySc34k3Gfx/ppclpRGlx91OllIpsEvvKeJASrou0UrOTO1pTTDht1WQU0YPBX8567jQoJnUOasMtTLSRs1WwKljxkkv10STFkxjAfGDYqWWfss20i9ZyF4hVtHIjWVKFktfLMquvuOgGtYybTRmPVobaD0+g6aj9mlJhfPZzWx6ddVKm5uSdEASiMrM/jgv+sVKLI3d9oqRYuICbJdDV7udq+s4NWkbtagpAFasrJWXpCwM6/byoosq4YXBGiTtrlAQaHVZ4N2ofJ704EAt9kZtAXyFOkxLJpCEWFITSRGcEUyTv3XrmQYA2QI8TUM74ntIc7qeGVpY1jjXzAsza7QKqoOlmSLbPfdufAR9ekL/f1S/9Cs=\", \"config/data.smoke.json\": \"eNqNVE1vEzEQvSP1P0QrDklVmjSlpXDpAYSExA1ubbCm9mQzitd2bW/S0PDfGXuzZlu1EtIexp4347dvPh6P3oxGlYIIAWP1afSYzocbocjzVTVdQ11rnJJxbZxKkkrZMJ+dfXznwN+3HHfSBS1Jo3AQI3qTAo+Pp8enBXPIHMHXGIW0um0SyrRav+gTEowiJoKBYTfVd7hDXZ2MKt0bnzWEkAzZG12CZP3srEWfOgeJpbeNSDQNNChoKRoKgUzND0TfYg8O0Dj+FVKvUCn+9NKPfBh9+5IOX7XdDky2CoPa29a9nPCmxC0WTzkE4dBnxoybz2az5P2TMVVwmoY1ix4oKTo7/TDrs2xAp2fIdo6ziyI1hvjsKiCq/Mr8slxxzoj1LlUT2mhF9xOwBY+loh7vW/IoQGuRK8GsEeRK9ASTtAPezqPzVmKvfM8fHxgvKQrlbS9Ulqdokh25dIcuy+7Oye7xr/2NuL0Ni8mS5Rx39jWpyfXbnuwT1Dh4uQ+29RInPXpMbu+sj5NXg1SIe8Xqkcm6/n9kpIbDuKwvuwM1pDX4VYyM6ADlz03boCcpbBt5CIWKO5c6ouIfhXg+L7UwYASZpXCWpcxlY7E1SFbMBoq04TIZJQzWkA+MJUNxJ7YUVyJFrxFdNpbWC031KtZ3TckfJDdUrlplrMFqUNaO2qCe3m679uUVkO7PLs+v3veJpG24D7gFcmdWv0NUw2QNNtbvBsm4Y/J6OYxeyT0YjTLqB9Lsh7VoWh2JGwvTLjs/LagGHgRsgHgxpHzAe8GDLJNyNZw0aNWTSbsDuUaThqUK99zj/2ahO4o7iNz/iSRjLgbclvygsDz/3obQDYiwG/Qa3LMd1CMPa+NVfKZ59Ia/v9Ulv/I=\", \"config/orchestration.json\": \"eNp1kU1LAzEQhu+F/oey5y4mWyvWo5deKnjyGqbJkI3dzK7JpCjif3d2u0URPAQS5nk/mHwuF6tVdQLvOzQnTIRd9bCqXCFPvnwgNfda6d1O33QFqD7L6YJv2R9jPUB6K8i1D9yWY31uqvXk5oAhI5vcl2TxHzsbrHN9buR1NarWo3pI/StaNgRx0h7G3Bc5hzF3//hUP8+5+5/cUZgKmeBGybWhyRCHDrWKplHNnbrX2wvKkLwUDIwJOPQkIq3UpX22Lboi24iBCmOW2WYeJbRIbIaSW+MLJPeLud3OTCEK5E2LkPiIIGtgELdWljFyenPhIryHWKQi5iwNDDBjHHhCtPrDMHgCSU4o1zRBs838c1cXDhH7Iploe3JTr02j1HLxtVx8AwDInzE=\", \"config/report.json\": \"eNpdj80OgjAQhO8kvAPx7AHx5+DLbNayhgbabtptMBre3VJR0R7nm+nMPsqiqjbsqdVKtLOgumh78G4Mm3PVHOv0ttlj8KZNNAkpYA88OIGAhgfKzrr+WunGA2oLKbKyrP4KHfJv0+7DmLyJgnmMJyaUHH5BcQzXJEVPAcTlFYnul+yFrOoM+j5pj1mZNRTVQdB3yjXNYbuAEVMRgxbyue218U0NYUgthqz8W+rZMi2XELX5/OZUFlNZPAHro2Ni\", \"config/train.json\": \"eNqFVdtOGzEQfUfiH1CemzYJF7V9gxYQKgXEpa1UVZZ3d7Lr4rWNLwGK+u+dsffGJlKlRFHmzHg8Z86MX7a3dnYmxurfkHumeA2TjzuT88DV9Bt+z0VZ+dOjr9Mrbh8C+Omp8FXIpqvF5E2MrHUBsouT5F5mdQPCkwEralCeWS2jR8YdSKGA5Vp5NDaeDqBAeDFbHCRDASuRx4jchMZJhZplWjs6LSjyn89mTSZu5TNzXhsjVInIkksHCRN1xiVXObCKq0ImfKK0gubcJXAfLDC8GZIgtBrhwQHjUjJvuVCY+9Ghg7cBhhQYbnlNwAsZ0RxvismYfzaxkDIrfDoRUZ0R4WIVkTpIL3LJnetwiRUpirbck8/s7Ww2b0EiAh1WQPl2O3PNn1gBxldonfZWvHPBPWf4i0HLSHOXhtdZwZmcpxRj8yKaXx1VEgdeM2ek8K+jKH8miL3F/n5rbMldWt5yO+9DMl6WVOX/UHigVF0i8FbkaPmZqGNSl1ITew2XDKzVdvKrDUhq6jrRSarh0lcWeEFk7vURHizWK7CH+bDdVJK2OQlYskfhYCOIImnBXomkRcdA8Uyi2FAw6/h4DgYkbIDG3G7yia23KHxdb4IrLFCXKF1mNBbkxB+603zxvu8Cyf8BF4JHrGDoW4xKJgrJnLxIAkTlvMuRzM30gILHVofDQ1ZgM+2Ef6ZQsv1tFgHe34EfDJaxgNzhVWJphbDUUh28Cd69I1vXW5RNEDTYfAmshlrbZ4arbinkuGtS8yKS7GneSrrExKSVh6zhD66PSd+kZGAZ93nVLoS92YeDNQ/SQYnbyrCc5xVg81G5cWy7oUJxox9/jOU01xrUj1H5vdFCDSnAf2BXXKZFGI/r2uVwLzBR18FHnUUPRiW7tZpzPOEewIycFoP0DpxLc/nST7moseOVDjZmXvRSoQXMMsAZQL6FCj5tqNGOSJfGqcNp6M9XQcph4t1BzjQzY90FQ21jTZfHaMwELd37vSY8qoCeIcyNj1Bkb1BCFvJ7bDqoFWngZpcd3X36cnzbNR/VtxRPA/zq+vjk7MdAcyVNYoMffr9h18enZ5cXnQPOu8x4fs/WPT8fnxzend+2EQM2cMGV6eF66R6IFUgKPLs4ueyfjeYBZvTw6qKbpe0t/PwDuM9R7g==\", \"config/train.smoke.json\": \"eNqFVdtuE0EMfUfiH6o8E0hSqIA3Lm1VUWhVrhJCo9ldZ3fI3DqXtKHi37Fn9tZNJaREUXzs8fj42HP3+NHBwcw68xvKwDRXMHt9MDuPXM+/4fdc1E04fftxfsnddYQwPxWhicV8u5o9SZHKVCD7OEnudaFaEG4tOKFAB+aMTB4F9yCFBlYaHdDYenqACuHVYnWUDRVsRZkiShtbJx0VK4zxdFrU5L9cLNpM3Mkd88FYK3SNyJpLDxkTquCS6xJYw3UlMz7TRkN77hp4iA4Y3gxJEEZP8OiBcSlZcFxozH3j0SG4CGMKLHdcEXBHRjSnm2IyFnY2FVIXVcgnImoKIlxsE6KiDKKU3Psel1iRpmjHA/ksni4Wyw4kItBhC5TvsDcrfssqsKFB63yw4p0rHjjDXwxaJ5r7NFwVFWdymVNMzatkvndUTRwEw7yVItyPovyFIPZWL150xo7cteMdt8shpOB1TVX+D4VrStUnguBEiZafmTomTS0NsddyycA542a/uoCspr4TvaRaLkPjgFdE5nKICOCwXoE9LMftppKMK0nAkt0IDw+CKJIOHJRIWvQMNC8kig0Fs49P52BEwgPQlNuHfFLrHQrfqIfgBgs0NUqXWYMFefGH7rRcvRy6QPK/xoUQEKsY+laTkolCMmcvkkCiss+Rze30gIabTofjQ7bgCuNF2CXxkvFvuwmwAA9hNFnWAZKHd0m1VcJRT00MNgb/jGxzr8wG+hajeqKg+eZrYAqUcTuGG28t5LR50vAqcR1o7Gq6y8zmzYfk4Q9ukdnQq2xgBQ9l0+2F54tXR3seJIcal5ZlJS8bQA2ggOGe4FDj6MdvUlHttUYsYFS5sUboMRH4D9yWy7wP03F91zyuByaUiiHJLXkwKtnv1VziCRsAO3FajdJ78D6P590w7EJh4xsTXcq8GhRDe5gVgKOAfAsdQ15Uk1WRL43Dh0MxnL9cjNMejjLmwammIxMtdY21Td6DUybYo5tMO0avEebGtyixN7pgEcsNNh30ljTw+ZC9/fruw/GXvvmowbW4HeGXV8cnZz9GmqtpIFv8zffP7Or49OziU++Al5QFLzds3/P98cmbr+dfuogRH7jn6vx+3fXvxBYkBZ59OrkYXo/2HWb0/pr0UqZTHj/Czz8f1lOa\", \"data.py\": \"eNrtfX9z20ay4P+u8nfAoureAxySlu2sN8tdOiXbStYvsuy1nJfb0vJQEAlKWJMAA4CWFJ3eZ7/+NYOZwYCknORe1dWlKokIzPT09PR09/R0N8IwfF9l67TKgmWWfkovsmGdLrLg1ZtXw9evy9OnB0/+HLxPq583WRPU62Xe1MGirILj/OKy+f7l29HDBw8ffLzM+F2Q10Fa1/lFkc2D8wwaZsEiS5sN/H9WFp+zqs7LIjD6B6/TJq0B9qyCdvASAH4orwAMdCky6BGcp8u0mGXzQXCVYS/8CyAUZbVKl/kv2XwUBO+rcr6ZYf9glhbBps4C+F92nc6awYN51mTVKi/yuslnwboq12WFTdNlUKer9TJTqNIcmry4AIjHaXWRBXmx3jSMDPSbZXWdzR+URaZJUpVXwUVVbtZB2gRp0OQrHHkeXFUAKSuAGgx1WK+zWb7A8dOqqUcPwjBE0i2qchUkyWKDNEqSIF8hcgCiKBuiR42t1NPqAnrXmX5wMdN/Xqb15TI/17//VZeF/rFKm0v9o6z1n+tl2sDMV/pB1cKufwa0s2ft76YCCuufOFNBf1YulxkRv1b4vyo3BRB9EMyzRbpZNvMcu1LredpkRCZpqn4PCOQvQFxpuAakYUaq3XuaA71pbtawSOrFYXEzCN7AaOn5EqC8Tdf4dhCcZrBCwDgGAYvNan2Da1KsWxrAcqXItsF63j68SStcWnyauk9Ha1l8fPtz+7beNPkSR3v44PT98ZuPycnh26PTYBJEYVOleREOgvAzsOycFhZ/NVndhPHDB28PT394/jW0LNajTV40z7+ODq6/c/6Bdt8fnRx9OPx49Do5PXz7/vgo+e4N/OfVu+Mf355A7zBhfk4WOfwnn4eeHh/e/eTpALPi9kcnr969htbHhy+Pjs2Gy/Q8W4Y8u9kSNnmAcoP3BJD7fQp77gNSvIYNGn2A5YfVPAJyVfH44YMA/gGO/5DmsIGCdAGrBbtlvqE1C+pyU82yIWIdnAPnzNPqJri6hO3TgGT5Ib24wEY4EOxvEDFFllbLm6AE6TCSffQQdvkiWJbpPAFBs8gvImSfMXJt8L+Jd+Jg+CJAPjyDZwPkmqkgxh0S7ABTxbbUOea3Vzk8NpqMynVWRGEF6wfcVc5h8pNw0yyG34QxcsQl8NMyE9D4T76wutebxSK/Hs1A6i3K5TyKgwnQd4T7NTR6tYgBTvhyhJOLGHrctsuWdeZ0a6ob5wmhwVx6k66W9svsepatm+ANvacVw2nAUw+QChcwMFc3Cv9x+PZYUIX1JBlcASPkVQY8coNv/xL8x+m7E1i2bA6LVwJs2A8gGoCQc6DhDRCO9jWM2UMAxHqEiinpUgHoC8IS+CIv6gY1RcTdBrTasTELxv4/0+VG4f7KRrsEOKtN3YBCQP1Rnv8L5Foo48ik5oDObThntYWbmOQ7/rE2NwQ+KDcNqA/8a5WtyuoG/0o3c2h9xyBXObUFiDXQHvaNGmM0zxeLrMra2cTtbKXXtoktwrcC2l6YWuT0OLgVKHd6etiiBlTOFkDjRoY9k+lNz4p0lU1jUv74J6jGwJByU41dWtxEnxGT4K+T4IDa80/owGPErL5ZLY3yerYs6yyqN6tI3g+CJ6ODQZCe10lTLidPsuGTp9vXkeTr41a4PkbJqqakVnRd1nmTf2YFDcMFTRk8CVu6qhnb6zgdXWRNFNYzAA4/4+APsF0LUFPhVow+XoKg0lbOOTALdM8YF5w7wstQIWUVGAViJNUGNqAG9BoII03PQhDUdbLOqgTNiBDWA4m8FRHuO7I6dkjS8jigUQglWrmaNuUqnyUohpI56FCQjze4EcdK27YyFVU+2FkFLcO4lbwnQDHB02iAqjQrmtHq0zyvIv5RTz5WG1Dk2TUYbEn5iX4Kfk2GUgrVw8QCg0I6Yckamc/5UfAVCNhmtQ5Nka5BiUC/2lugkzQ2yTCQNgigRjsurWd5PvkuBdk8gIUEIddMng5okyefspvanBL+w91HaDRmUfjPQiFa1iPgxWUKUkCjaxE4btdoDhsJNWIi1gkZAXWEgioB4pqqcIC2Fajggh7S8iyB1mf4TilFkoWiDRUMW+BiixEtUh11d+d3MPpJ2XyH6lzJJGXrAyiQQyAQg3mZ1QSM4IBUQqBaJNEMWuFIChrlCf2Rs7weXSzL80jmEyNua5YqNP8otnEmiHvhelJqG5/RAGE1uwTpfytj/aG6C6A9GDI20rKFqFO7OnkB4hxMk+VmVUT8PxDCykTF7QM7R8nqbK5WC3fNAI80cxRsmduFlq5tKRNDs4LVFLySwWLD4CDg6jkRlP9Gkgpqd45QJJxasi2QUjAAj0QCkkGqtuZwsWUKcVeQjQbCW9TYKw1SIXnbjoJLcJUyAxHc0BhLrQM+V3SpWlLSZFu67p6barvH1BAf3/Q6GMmTEzr0KFYBTQG7I6FTpXBM3c8yPtaw2kzb/Y0/f3Mesaia4Pbuoywcxjdw5kYLw6Yt2hMWWS0rwwI+tSguEFGfp8tlxD1a+ltguG3sX5MzjYanz9RarLOpsVR0cFHHLbDe4DwN2jTRZw+iPWjyMUFoXRZABKuxFvPhP/+JVuJjR5oAiBGayMn5DdA0ksP+6HyZfsqenkeGL4RUGMARBYZG8AXoi6SGt5Nv4hH/jOBFeJ6DOfPgAWv45bKcIY1N94g6Hf68KUFwR4TQ+vKmzsF2wSNj7eh/wBJYskHPSQOWW6NbwYsHzvkL247VufBQRmePDRztOjBYZ6MTZwWGm+XDqXA/oamboczAA6GSXV0oZC1tMZbErB91e3YtJjZI2hYTtCkjNNvI3o1t09ei24geg9qMNar74/iKnEPKdYVurtW6uQnUiUTwogkATkAywqkzI+AA4wd14nVGoaCHZ04WaI/IKKX2wePHJs5KKw8C+tmZL9g1K5gugb0Txl6Byc6HHwE/JAIyEg6FqrT4RLtGTIEWP3OYgX4MRtZkma7O56nMIBpG1iSsfmfYBuTk/wgs8pATgGHGWtRp24MwOhvreUzbNeMpMNTgq0nwhN6kdZ2hH80zR3QBMHpmSxRpB8gOFjz43a7CnnSPTVnC0GTfW75R2xtURyLXaP8O3J1PzwiW/F1nGTelnV6sR+jHqdIbvctPSbG1jllAaA3nHnYB5TjyEKg6L1eKt3HPw/ElEOG4wpOCsb1xF7T0wT9sFLfsoVMegDuqrX2eNVdZVgS/ZFVJOgW9TwokeXpn6NYM9abl7pP+cZngQIoUJnaR2VwHorm5WWcT7fNjuEhG0HSswdklyFQZncIbpdqjs1YPAjdgp5iEqlq04N+C1oUor/S7Fy+CZ09juwnBmwqn0LZsRxYvbgLPIws/i7GgPW7PCFqNZpdlPutMmJQQUW2gFlWdkupLOKot5Wccj9IaidNudE0lNJPXN9xO9iaz8vkmX86ZhWE7JgC9iNjeby0jOuEM6OAwlqOQyM1ktrjwHGd7tZbmC1dajrRib8oIB8K5JKg2rtGsIuHz80jOFnjyYGfjaJU1KeIyKjYrQya7codmZMhR3ip8tCcH6sSckfgunDaWjkgsFdbp22nUsn9naNP0ov3TGcA0jrdszw5kPzS8lllt0EQDkyG7ni03dauZvfrfPz6z7z4mkFp13k9dnboXbcamCUu712kcOzbAdlulM1SPqWJM9ZaVItoGvDG9WGxTL0qz3JkCQL3U9nFCzsNVfv38azaKYC+2isHVE2zx/gJcyG2VDHC3vuE8IasQ3oPpVzewdhF6QCZhfgEWMczbMPYRbPRL8L/wPy9eGBctzw5imOkj6+rl5Xd//ObrPz1//eTV0ddHf3z553gPOE//1IXz569fH3z955cvnzx79uTJk6OX3eOpH6UnCOrfAr4WMo4bSM0E7fs6odMAngPkOGCTdiC+T0P0kVd3ukVNK5f0NRla5tIx/L7lYIkN02gnQOpIJrspcrZDAa4BgtABnRc8Dtjf/PTRI/ipHD+wnHgGpzc8lbODqbyts1nJR3Rq9pXd7MnUPjrBWFeXWZVFhMhfudMgOBh03jDcQfBkEDxtNRC0gkl9Y3ja+IabrCRjRRxrSWyo32ZRZCp96y9j3WOBBNdYocQYGHNk74PAX1RwHh4H6/kIPXjf4a9BYPknfH6pzizgIF8WojMJ5AgE7tl4QM6JyIIXT9VkQoBHfvcR4Lws0ij868nhC+d4DJjh3esI8U34Qjfhy5tID8qO2Gtlb6CkvygrNEhInoyaMqHr4ci1ylqiJBoYX4eyTBujoaA8cWPtOAOc8hrw5cPgVhe9uquREwrBJl80XaJn2q3FLybkmmGwI6TOOrKdnNRs64BHdGLcNZwQl96bF79HaL0fc7TIIV5nlVV7xfs6rz8Nz9MZHtfIzA/oxisogaeyIgN2gxdPnn4zPM/1GfbN65o0PAdSiLSRi12+MADaJ3BIaZIEZMtywVYbXqqIq4VNuXN00bbHE/fegUCZ/e57+yA0tmF4PODdgTYF2KSfIgMOTmME8qbgGzlcVQ64UA8jC0J/T0Agm23w3uD9h8Pv3x4G/yph1YBFVyAnJj8dHof36FvfFLPLqizKTT05effh7X697ZmHrz4cHX48Cj4evjw+CnK8/AAzJKuD6BPswODj0f/8GJy8g39/PD4eqPc3wcvjdy+N5xxP9Obk49H3Rx+M56Ez2PsPb94efvhH8MPRPwh+CxFU6E9vPv7t3Y8fgw/vfnrz2uj55XM6evteJkZsnRDLBZE9CwOn/XFomTfgE377wGY/t7VzA+jf8YTtiFksMTr3XQRqtMT+nVVlXbO6A/QOnEYiu7e2EUDzDbxH4VsnaLzlhdnB2O7pfM54yobHtSWX6sBgqbGO+zkjx+h0YOjIXhmQYx8QWkhn+DNq4RmTv7pEUxo3v0NbQmrMDvUGJqOGnqJje7pPJAZa1Yn4kC6yyFnP2NNDjzuC82lWzKOoyK6bSE0kHpg3EUZMx2lTrolCdBnbhbsGcW4/FRXCk+x2OK9A7NuPe0XK66PjI9gy331499bcLGG8V/8VxhGEb05Ojz58DN59CN58D/LoCAXCOxOa3nlx8J+Hxz8enQbRt3EoqsAZifhTNtd+2562/inM49XH4NW7H08+Ro/izoSCw1MezpVN1Ps/3r05MYUgNP5UlFdF8O6E/xghZ0++DQ5PXssDNaUJL7mWLT7wP/3tCKjC/Yj1//ri23DQbahkI05d75A4dlqCkZXBkLBdori1t/VK4a3I/+Pkm/xu1IOdRfoPo71YFoaeDdYnc7+aMPtu79EnXKE3Lp4jI7qRY1sEuh+Be/CBfyu3S+tob5HksXfZhKe+NZt/25U0X76S22c5K1ervIliU2GhvhIPimisPc6D+yirlvcnQcRBuGC3zj5F4Yu//z20fL7i5uWhxH3DPzhagjBwFbyhaRVfmirWRLIzYWITNV+fF2KPCcLhG06OEvtKf+/lb4j3JJFQiI9Lzp0cD7eNIDTB/ehRZfVm2QgteP+kV3CkGAfnZbnsjz5tnbgJqmO+5eqzvCyzigfRnSJU3MbAFGbXZ55ZgNTNlIkF8G/4OmuA4zHmpQehO/MEBwRdAQVy/M2ZAmFnCBNlzwgdRO+4xw7gyqvp2OyrrLks5+E4COkgmoj1u67yVVrdYCCWKyFkA8A2MZFI0Km4TNE5gdc/477l6Qem5bIllr3wekS4C7tDKT+SnWb9OPIKgXiT5UEQFkN4MfD0stbY6eQH3R7+e7rpUWroeutRCBT/meTov5AQ16QNBk3yOqE78h1T6oOEsaT3gmEO/aWA7ozfd6ackWhZWF2fNO2oKmodm+6b1+aVb58bhx5gnEW5IJmZ2jfFWu8Om3LIImWxKWaSwMNQKCsITUaREdoQA3Ctp4dShpZX6Q1G2MHhZh6c39DFK3fNsnkmwVLUWo2CwZRwekOfIp1XsjbvqCmD5qoMVAS1SlsaBRiSy6DSzyVowiBlmTI8z5dLgDnEK97Tvx/nFB02z67lELgG0Z5Vn8lDdykxKuLRxI0l87/Y4CVvk2UjTcT/G/aBE0L2W2vnHvC/pbLDM3avUrNf9gh6LVXIdderBZxYB2ItcnEniqsSYvgv1AoHXyz/D+4v3Q/+mwV5V/vdZza7ZtGjBPuBE/OFzH2RaQP9f01iaRICOrvMZp9+9fw9e23fafd13Xe2nv5fqC69AbgJZfnhgJKeAeerbAmCep2OvsO/CBKymn09B6/xwFJj8Dnif5FV3JMekzluNqG7P1Aq29rgIBnFsegmBp7zqlwnVZbWwM6RymXzxwnzXY8dVr7tBg+UA7DJKqVJn9KfQmErO8aXAPKQQz3ZN6rjjgeGQvD9Vhoiu8atTw7hW+Oey0iXs8MqTWw4YkVBYPLI9OAMdzaNJWRZIvgp2anK8Di/xjgcAgp6eZle1BN4zn6KV4enR7sHpbEwbDhRwHlEcQgzR9Gy4sR4OQu6VcVJmhyAI9FvHIkXQUVa49MErCQXBDfv74jIwQLbBEcgCrBkH43bMHHDme0N/nbzG+m1jmJ078Jw/DNuhJAlqEi6hWYyowZmrnheaLbYBVm1UwFBczYorSw4/4A4ir0jdgxFjR/DLr4S0xaIX6EsAPkHo5YF4IC0s/LqdYq5iwQ59FUey0j+rwJw6G9kPMW4nOZCTessrWaXKlQ/HtDmjrekISh4Ex7z1w+5nU6LsKXOY0wbrRtQVo+FKhxqr9ORVHLNXYc8MJceuYw7Qo0Xj3chU4Cdr7LuVIIP3fgHm6LerNcUX4wso1L2xrLXzGFoq9o4dnypalOpKxuhlp0kJG0GjOjAlBOtnAdZQ2cFCfiK2tyo7ZGNAp2tLTlm8Ebwie02mlBFT/qi+x/4bHuZ0qys5kqG2C1McWKFGx6Yz1AOV5kY/gcPH3hDHw0qqzT8SeALqHzojxGXCwzp64m6NI0DDuTEAXYFdpoHcPJomYNZVD1TYKZGp/yXTCOGOeJN2kQY3EFpGwZKBvG+mthjuY0MamJTgGPOjNZKsWb3PNVcgpmlEB04b01yQjM7xtc16E0U2SvV/u4YpqKrx/1LpOKD3K64bfi86/SWdZWXcXdIRSVOrUEUgVSmNSnkF6EBYqCp8mtuLWtmrMoje9dxPCDJl6gnvfgsVJA5Uzeh1uE05hDLdv0pDidJizke8nqHj/R4HLj27GkLKPgqeGpi1MY4tWPJYJhiusJYnWTN5WHaIb2E+MqDnhhbIOgT8d2iglRRfYockhoPdFhitvTF+SpxuoTTDlJ0yKAdBU2FDL0oP3JxEHA8Mu5uqtwx+pxXGEuc8HMVrINhwVckknAQfjdKP6f5khIXHvVOZ5VeJ7pdUqWrZFGldKCBCdl6wNiAoZL2UiwHGPKJwY2hmXKmjtHLTCLdTfYOmS3cDdgyi9nWYlpoZf32DO/ZNq7UMXu1S+NjHujse+wHoNeWPQkKQN/i+4FoTrP4yAPMem/C6uXVsctsZi9kAqaTGsrgKXoRu81bDqIuSdokwiF2b93OgiDM285DjSsvrKXF8hpNiWsAa0HphAmoX02sfsJgCJIHoKouhDSRP0fqj8hCc7beWI1YPJZVxAdi9XyVzi5zvF23WBcZn7QV6TR5dWd6vE+RVcBAaH7CBPtq7IkqtD3oBnzDoGofKi6nEgdo5ZjtjToLbHQZZiLGChLVsAiUY19pU65jm4H9xGd2EEJ0dDduT9dc8wbIj0GXFIvqbdx7WSCJ/vi/zjWCniU0MH51IJilJSY2Cdwotw165a3DKNmLZmwxmYtG4agIW8ReQMrKknpTnQBLobMzmGOc7hgOTVC+O5O8BLRFqaG9nMiqt3exyowYe2IIqN0ZASOTmKD4skV5oKlnNihBz4GCxqRvCSCrgc4wcYu/U7LlziWWw0swgvPEumBg05FvAAQBIqknWNwb17CQGHBy6nmzsv2sI/NStisB2cIcitpgA5OadJrreMpujxcTD3v7VnWx3NSXHE5iRQTIc4dCPcSglN7uNHcQBjblOXkbJkhwsEFmqQRn2WDg9EYpMQnHwDsxzXzNheRR8OL9Z59w1RLZIQrAKKfYfg+M6QijOCLXbWDMxIbhETHj6Qiv5BqeTYRnZ3dKHhrghlOgp0hye8J4hLcCOPv4YmJ3NK6uZiMphWeHBikKfelW4R2vwkJMISAobRETBvs/8Z5sqcBLKCEVmI3DouJxsAgRzvCWAY0Pns/vVOE7w/Fh1KIJJoZCeazH8Lb9ksj736b2D0l0Ejxg6sh8zOI6dpJIq/Mmrko04KkTatcToQG3+Z0P3ey/rgsCS0y1wsob222VYFuEypMFhwYsMsnrsQCLMGO/8K1G5M4kxD71hQxvCznlXe+A4QQZ0Dm8VicSwh7rTIjNaS2O5eG486ltR9DjM4/gdpVWv9/T19xkmIFvSl2BnmBaft99Up+uHXdiQXfL+t9Ovu8hr30Rj13xOe1t5UjIg3tGrLSUjXtLTJ42MEEj0UiKR4JWGPKp1L6XCSghNNjQ35wMNqxB9QAxT58FP3443pli5NYxAwbZFBSxwZcoz5Jeb6qXN6hWKt69rkvY96og4+kzmphCRJPElqzGL9f6JpzQ7KY/3KyLZ/BGhog0xlioDIk3p7iCKp9nimVaanzKboQQalu05kub6GZmJ/J4hJDqLNhhqJ+1OI9v9V5bcgrbvz/+95hkkxVkwmgrzrHtdrvoFV9NlhconXUEJSCTwW7Hgm2MDuIVqmZcc9Mp4oTCQzXw1wriy+LOjpdDAF0PCny6GaTH4YCOBqrgh3tVQE7uioSGActXf7NV3HhviM3OWCBP+/Y7EGFeXhUk9qgkmiaEAhYPuhzX6u+BLnzpSYWzp4o7boNWBMa4o6pcZk0nyt2sr+SLE6dic+Li4BUCFcLOxRVvM/3UvodVTyUWYpUW+QJmxM+d+O7xvYlVsDLzEKqgdNg+IqnYACGUyd+ii3ARZaeZyYzdXecTnICwgOEajRGn63fXOPaMDAPni3TWHf2Lh6PbCyKVNV6dfqayCkQCPVpfCUnf6FKc10N+ZzsbIcdu0Uo6bcuwqrzPPefnkR3GLOFEgg7ez9rWp/0g6kLtB2YsVUCHvYHtE+/cS4qGQePLvfUg6aqCfUXWujcwvCnHgk3nAmRTofGd5CpNzBdkZW20xJmJ+CGdp/F2GMa8pb/xpNN3s56T1zFF37QqGD4qyqtI1QwfbZpZPMrrEh2FaFra8UB7sBGvXAJE3MlIsiD3ZyF4BzRAU4Q0ZGfM2IzuwRURP//lpvhUR6Z0sNz6REBRh8RAOiuSg3DsxH28x1W25j5XqOViUWdGRqdZuctzA6mSTjjEV2c59tzH2U6QFrRbSqZ940l3dbIS6eBBl7Y8JNkAesxI/xXzERCrBUTxnni0B5vgRdvMVXGCAJ8z2RPS1gkT/4d7HJWbTTSzzZKMkuDvROFYXaYer5q8GulIyjmctjClMt6KqxRk+K9eIJ+ybD0JqYZGGBsD33AIEnMYc4xxzGQO6nfBeendsb5Um6ENpw2ToIsfHSZhFnf3x6kRbnLp6lTI9cS6JWSNjj2HEq/DfQ5ch/EjyaqEGWHZiDFfFXZa94dL22cActW0+Mbdasr7ulOMKk/kb+OrS1VBUJZVQnWtJlIM3C7Pu6XysAzCsBHpcGpV3Dqj6xsVM6eNWLf4sAeM1rwc6Ub7xltXi99z7FA3nGRqhk1igVHegFiYlPqRMSOF9vhJG13Xdl7BGifGBib71jR0pcPZwZQBUvFSXbFU+fvbO4C6DbUjpNRW02UdnXrCJgKD3iJeEm8XO0tgvU3a0q16OXQpLVW3l0Qh7tPOSGyoU3EapC9FROaLRIrNg3nO2Xtbq5OclIEVICiFdXFMBXM4zyrQnHOOOuBKYMCGdKZVDi4rmE/rqk5d3R2kM5ubtKHozpY+erOokxAWOrm4kbOQGaSuadcXbNgliNFdvEtY9Vh/j+B8gyXljUrNkl6jpn6V4Wd4zPrIJsCJGTSvwmu4RH7vnIA5b61J0YcNQJuaz5zgUo5xU+HKAyck1hfL3N1bKo7ZiV0eGPtr0FfGX65BfTGtUlQUi6ltCpCj0SNz69nfYjDRiU0+NidghomyQyGrMRMJA1lBUN5I4eD2w0qEfmiXKldBuTu2ivMJAb3qrC/lWytSHA/THlrBaUalIRv01U/kqEJrb8SiHkspOeiGKFr9SFnLbGK1PgN7fAvgmQozoTAlkEMwghUZ6bfKubPluEPDvutQ0CRYwPBZta7AJE7kYGifskJGFk4dgrUn8ODMtlN8SRXiMt8ZRehLihCP+j3qRvqgmPEmvhBDN53C56Ay1Gb7euoJI6rbCCIrAIc3b6jqCZuvrP2sc4/U/tYxHc6iwWKpUtj1Zfr0j88jz1casDJbZ5ndjzHgiQrMF6ymUk+icIASbQznMbeetkiR0WV2rSppK1NR+x49VuNI+TL5CwXdBtZBg64iWzej4WcjuWw73YzJ8cdJjAdbZceHbFU2rtPc8E/rbzPQlw8o9VGcBUoSPLbD6uN90N3pIxSHmf7kUR1Fzob2OvhiPuY1GNLe+YyH3vDuR260itvv8zZsbWDqIMb9ZDpVAkWePU2jpZgKFl1ohbFXbNnaGNBR2xKIMzxUqIceRQWbME2N3uJpxufsmIRfvdOy9pXI3Lz4DEtcVje9kdxUrq8HEweGZSY5HiKhmx+Q62My4dAFFrrDnLiuyHvVbEiZ9hsV5id1jDtdsHiN5vt9gac3vGvin5lxH9AfxzXxyRDDX3wvKTKwTB9VIGX2KaMimiZR+ONU8sUjaUIudaqcEBr2rgNl4mbXciKt+fEHyTCnALXJtiR0tSco78NzFeOp5s052yqlhIYiYcbJ4ZisvMr+wl+FpDpogvfEh7Me3croEPQBc0/1Q+cawxFXI8ErMWuwSVkSh/BnYadCm3WxEz/8b635rC02zJ2WgGj/wKpROGhPETA5fDJtiyTv4GDrZq+7HQx3e8i+Hz5hSnSO7Yw2YqTVp6Ss2mzb8kzumwvSuevXnelQb18D+O78C2CfjVOtSZVun/R9dCS2k7x8OS/3LDvuS2fBktFOUfee2tG7CoRr8eTWGFsveTgshyBOMpegexb27ge69bMH3vJ7Qm7nMwiDrfk9A3vHdODapGbFor4yi/72jg1u+VeRm/yXBM71QNylSEuJ9uMEDFVD/8oK4vF+qsANaXHIvHVNyJzKFo3wJn47gLIK1VnchjUwahczfkhcjFsIEUgYx94BKgxd/9IRLArsNZymel7I9cfEJcoZ9h8TXlMvDFXDl2N5bXCxn4ysc5aKO2Sk3qZKvngbcB3QsuBSWp35DLvr4BTKdsp0+fdS935Ej6uuR/zrqdm2g5qPIW2nZU9dTfEgUoDVaVZhJbozOSFnq2nwyLMNdE3qLsSeonrWQATsjJGbjlbpOnILS8ed6tddkMYJY8T3oxE/UlXU8HqLTPXY0ztfWGVb/BhzzRjAuFsN3PHAxX4AVKocV2tr4fq2CLlRls+0F+5FZo00fhYAY8ujg6742o7u9gLveod9Ad7KWTexgn4jjtqT7YB/x/6qscrHlxc+96D5j9wi0Dhc2ZwTpon1pO90wJ/MqidwBsrApgh9VdClUH/32wv+Ic/+C+VAvcBQOimpUMdTXo4iLfxdBT2NWOcTEPfNneyl/Nm2T2gLmovNchmZu14v/G426h2m/e72lFNkkIP6+8sHt6mkBPk/ZW/r9enhEOTUQRuCmhE8lA3Gqb+vwrAp+2nh0wakEn4NKZKdMQmk7OEOzaUh9Q3Vb+c6B66RWTfL3YCG1hjswGy3sHMHlepZLE72HUmrJS3sZd1cJSd8jaFdPZAMn46KhBYx7wyyTcy37hIVRi1xv8JnHbz8eRUeyI612nO37zFPGGlDiColQDymMevCsNMrLFo5XqeevHZvbruniX3ASi6za/Rnh7eqYNvBk+fXd77isjtS4z3t4bxQZHPjAKJ89L1HCv8FQbsSODfrgd3jLu4cTu2F/MP2kgXjfT4J7zf6+KNr/E1yTI1f5TV7lilBQQ1xRx8BNlG6Q5xuexC6C3ecqoT7rQBzJ8LePI/j1m+P1E7hFPFF+a+K2ksQikDryd/uhuWZvgsfNMO9PzZ9+77G3cg8OezsCM7jlGbDZa07Gg+9nSzXs7eimBZmnIYZqeOXR76prxX25yx2wd9t2cGtG3rckRHe7UuuWZiF4Cu8Y2Vt9mZT+gD+urhFz0XcvZxmW51nVkSuy9nxnkB+nQeuBU01rMP3krlPt0ZzSu+4MS6k4IVcm9/6Brh7fNuOQFJESZSw65HrBmd1P16HS6VfR5QR6YnpetgjEK0osffpps4+YH53jbz/sEdAWn2CNXaCOaeLDOjA9/h7zFwRiSVAdygz75P5W9eIbf0y7EifaINI6n2alRX1ZSx+fa/rKTcrz7bVSySCHyUnf6VICq3irqK/xD6wbVePLFKl3DpRANY4nVgATx6BERGyI3PLTNPwiIZuma1LO3EHEDATNvxxmHBeo4tMlZrohBjrpmdHJ6/evYbzxfHhy6Nj42jBp8h1uY6sM0RMjgZr/o7n5tlTZxwz4ZI8ImamJbtIelMsvzjNctf9nD2EP4HRQzMmPEdBTNV87BiI30nSmlke4px1M3fcCmrKWNaROqQrEUe6HBbtRGbaZuXVpPrLwlvS/x+6XyU2hrE/TUzftQb6nrm2JqtDfMVhVLaKjR+6XzOmIkQvggNGCn/ojCdnWMwY46+UMe5zKRSvlDO3VyXYJLzQIE5XZJhhXB6S7aZUN8pPIvDQrEwoYTGrkyydXUq12gEVKjWPwEIK5CCMxRa07Vm/YkB6TpQ2iEoJsFuVKgKwBkJIizudxGiHckycskIma/YUF1ogYaXSLhdyDbvxOyqM1BfGw/KFGQFhSFPDGUvXPqEZKUziyV+MiHyL6OItqznV1vFFFanmEiyG9fr4yTi4p9uoz8d2Z9WqkQ8H4rHOiHc6swKh8Oi8BgXdBjSd3YaabvwHZktTuCAdR/GPOyMucSDPjPhEM8JQ28rWsEWKlXMXySUYMHhYCreRQNquy2U+uwktODVMjrtjvUT/6rAXgpmo2ypfnadwUANxYODSbWbpo0SqGnmz/axQFU/snzG7jtb35AmqEmASQ+RsFvW4b5+oS2+8+Q25hCtdArvhgTqwfbwt6N0uwiTHXPx6H3Szz9LdIIv2Q8X8jQu6Eg3vdV/a+ShQEIXoAg37rnDb69uwXpWf8ByRgRnghhqQsTlwpmZGJIw70AcdWWPEIIy70xh4CEfO97ETq7CLBkawjG8tVAl5ZzXeZ9WQiMJfh6YAkyUOUzdDDr8FqRWYq6Q+Mo2MeX7j+bj8X4KwU9iQw9vlIhE/WoNnRTgSoDkGAkx9SX2FkXmc7p5ish8mxyN2w3qdzbAqhHxNQX2HE2z0+zGK58reXF/SgCCfeONbO95bvq7efio3y+dndJ3AG9HNgaTirCRht7n2B8E2j/ygA5MGxgm8PD784ejp+fD51/gJ0wJP6VgZba429FAHd5Ax56tLyYDQjz7EorJze9mluguIefrqhR0YTvvETGW0CCSa+na/Lx4YH90IHZe47Ga7OdN/+OY1f3mDwv+xii9Vx+VTmij67TX5jV89XwbYGvVrlBKlje2/a+vQnW7mkDhSIvte4Z6dkvlkbSK/trZnJ5GW2L/1hP1mPq8770ggNcX+4xtVxQzysJN9m1UoNbAsYrbAQiGtkVW0Ss73/YxOT136UKs0sEmafgAFDpSKhZzMc7zSPt/I5zboyyq+b3dYnG5F0FF5GfptFRv8DXx3fdaFMgc6hoU/Evnh/ePbzPCz7dGEnvIIW9v321I9HXxFFbZ28BLB6OLeAW45JOuCBG4F550ezI5W2OWENxzwOgq9qwb2871/sd/93j73jr/995My9/eh/6b+81+b8x//rsGjv7Mb+UtcyI73WE9CedL/EqiarYgjmFRZVfHXstJgAXvoMvghvbjAz+uJr837SUzJzlCb3ci+TqsaBHR1AWyG+czwFz0anWC+6xrs07bkQE3R+7rFYXWxQeP1Pb3BbI9Zla9RvEySZF7OkiQ2u9LtfCp9onA4lBytgSriOZGsrccU12rK5B4A2G6IWcUtCCqlsLUTuw6cbuK2rGnoHaPK4We4llMEgKHwEqoLYuGxDQpb7EOy2Id0TvoiMPWzLhl306DaFMN8fq8uq/Q6X21WQzi5VBpXss5aKAejgx2ka8r1kM2RIdirG07I9YF61sLSX+khkCa/tlyMZzbm3zZ7CttwNQmjQ1taAGM8KHWDfkXYYMR/G9kS+FDlrveYAJ0sfDPbnbdLC8OGvC3OemzKK3/rvT4g3+XXrd+O98ymc/7Xs3LfbIPS9QtMzSBxNck9I8T7mxNRdtKks/s8RLk3TawT9t4k8M+D7w/I8pzYKYVMeVURDr3L9ERK3HVqonWb66edLr0cpIQMaUYlPjTJ8PMry5x0EkziMqtMhoKhQRWQBICZGNmBVCTCxm5XLmCbD8Z06Vb0MKAaBTcG5nwHJkqwks/05ZRS716Ki/xLSP7hXYj5KWQqYKDW0m75KHj2/ACEWTCUpQARqE5kIgKhzXNoYS2dgNx3i3flql6dJSayNZdpEbhCPDYz4NClCbxn0KCsR1nxOa9gxejW5P2b90fHb06OktOj09M3706S10eHr+nB0ft3r/4W2iZWB6IzDU0zzEzhHwMQ49cRUGIgx/4ODJDwbJjhf9AsNjMHNeId0+0rGc2XQGR4kt1CNCrh3WGnAfPfQA+pMs+uZ9m62Wb9BWmNraz8KHKLNBXMdRZ3y8/96Y/qhgjbGanRt12Hm5qJOLa9O6H/8NtxUCXKd6Ia6ioyZ+JWmfYe9dtvUbadbV/A9Ex9/VNBuePL5aKZPHW+snHAej7HSp9UmCShxMUkQa2fJCpbkTfG6Q3G/x9d503ERgHA+j+qycI6\", \"make_report.py\": \"eNq9Pf2P2zayvxfo/6AT0Aep1Xp3m7a4M+ripWnSF1zbBGl79x4MQ9DatFcXWVIleTfbvP3f38zwa0hRXm/Te+lHZJEcDofDmeHMkIrj+I24OpTVJvqh3F0P33/7Y7QXQ1eu+ywS79qqKOviqqzK4S4aiqtKwOui3kRFVUXbcnfoRB9tu2YfFd1Qbov10M/iOP74o48/ord5vj0MUCnPo3LfNt0AjetmKIayqXuspd92u7boemFe7Nbm8V99U5sfVbPblfXO/G568wi4Dtum25sXg9i327KyQIdyb38cDuVGYblu6kG8G6rySmOp3uyLutiJTlVri+Ga1XkNP1XJcNcCUrrgaX2XRc+AQkivLHo5iK4Ymi6LfizalpCnRoeuAmgzGrduCu8UHQye9WHf3kVFH9WtHSpMAbyBf9uNfdkfhrJSwPu3lSi6eqbmUsNPPv4ogj/Fen3oivVd3q+bTmTq5Q3guRN524l12cMEOaVXRVXUa7HJg23XVdH35bZc08zmncDeVNn20qkKM5hXTd+rn/tiGK7FbZ9DlW7diK16b7GAB6BlviUgeX9oGewwrqqF86pZ58Vhbd6lhkXX12L9tm3KetBE+vnJzwPWimDW9uU6RwbMNzANWdRfF59/+VUuuYqa35S/63Y7UeNMCyiuC0BYLg/s6OOPfnj1/ffP30QLzcCznRh+gEfRJfG+eCsUyWLA66laSchBV8X6LTTSzLRcItMBGkO3yqKfmlqsJPiN2MKoiw3hmiCjzok/0+jsG+THuaTCbTlcExvPmlbUiajXzQaQWcSHYXv21zhFjroG3qqEaiCpCSu4pmU4q5pik8gaqe0Z36oBwDTWMPCkOwDJyo5hsSnXwxIQzxCflYK/LloUDxsYo2oRnUexhBHjowN1hjjEsmXfHLq1gHYGRLk1zzPxruyHPkkjUcHiQhySnGYtz9MZTEpT3YgkxbUnYN5DXfK+FAUsgWXnjAJNtxHQb07rIK+LveiTvVztc73s5eCBz1ZEjwowxFeaFNQIxrPEhwgEGb3JIpCcNYy2G8RGg5yVINpgdFn0VtwtqmJ/tSkifDdH8Ak+LS9XabqSkIEwqj2W3hTVQaTUAT0ieA2XXgDgNPrLghBMuqLeiaQCZiH80jTlnFGUQN1/YKPnXdcAKwOXiirX4JBW0cvv+mh/6IfoSpBYhVXRHJTS+F10DbI8IzJ1YwkLyqLc3jGOzmCSB7Frurt5RARdq2Uyj0YL539piRC18WFuyGHa2LHoV9SV7UTNMc1r9F0JkmX4+cmbQ61aIop5XtblkOdJL6ptBiK8JMz8bklcoWxHZtdyPoHaqS0H1GSVWQ9iCdgApiHun8QRckMz6MJaDFWz9l7SsoZ+yzaJz2M+TcGp2sYva5jucgPiDtde9Oubl/PoPeBzHzOMcEyzq8P6rRgAbad/rxKI4m35zlby8LG1h+7OQ07Jz6tmaJ7YEvFuLdohekmFhDSKJ3gbHBrMCap3xYcEKip74KrfDiVKB+T3AhbSk/n5OY13Q5MJUwxij5gRIHOhtwOtAqNp+pmob8oOhB+I7CR++s+f8zfPv3/56idoBzBD5d89f/H01x9+MfU8Sq2rEqXOQg5Y/UxwnjPVL0mQhcIBOiE2Rj7UPLdpbmuSxZLnNqIfQOeg7mUSd8R8O6wD4BYcD0Q7N2WwhGHZ583Vv4A4fX7zOcd+2xxq5N4L/qpDwCRFDJCZehLJt8Q5C8ZFWfSaGGWxjd8zvrkfsyyCRlGmQAtJ32domdVDD7Rarvwm+AdEIg5w6KQcjP8u7uJVOq7XCbAXyxtUIdBkiSKO4ZPOV7MqxL1sqeLiM1AAWf0MDLHpUdWGFqKRNygK64MYl8LSKirAik0qqCUNfKK+UmWz/Vvg60T+6Be/dAeBZjxN6Vv6GRgKZwbNWKQsE2fegEpkeSTUYRoAJPnjs0V06Yg0pBMVBRfuC+jop2Z4gRW0aPqpiRQHKpigVZpbuXjfM6Tuz30e4ovk0NJIpBWnVsqkIgmtGGnHSYby2fW9bg2PJOpwyd7HTMwJFFxFd2cBGHj3s2Hfnr3HLcgM//cF2CLX4h1vTmZpf9gjL1ujk7QTlyfl74JELsnaYgA4MNX49pi85ROuiCSne+hkB1nkzLwzlCx6/m7oiqfdrl+8j38UQ7EphiIG3RFLROFRI39/73GJgeTJoGs0reSUB2UGLOKFg4UHF7gMbRtTZanlxA+i3g3XIABQlRK1YKGaalKkmDFk0fv7VL5TQ6FmejSBlSxZ+OUrpXZ+McNDpSrU1lZaJdG2ABpvDLRzwgb2XGbb5IsZTqB1095pAo3RmCKZYbhs3OYZQPyZ7FiYRgkAZs4BQbJz7k7/PWxjFb2kOQQyaRE/e/X6f2KvE280hMsfmHYzhvCUU/Hx6aYqf/ZUv6DhfOg0E2qVvz7HK3YkpkUlYKP5AStGWVfP6S+0G8Ydyj3r7LboarDmUfUeqo3SevsGdJ5dzYoIKOk+QdXs94z//KfvVEEZDdt/IA3Yob/DFvhQ232jFsnae7IcQFIJtf+dsPRXK2viE5oSGErGblAamXSIo5W1vqVdomrCNokjVSb1rd5hnqjTpM60de9KAbSkt3Ij7++39aZo3wwo4dnWw+DI9vTa1TUzEug7bd4mUmMtwLDbXQ+7q/2ZnKwzud03MzVFEVMhpc3xoY69isrmSPkIEGtjSxwfvKrsKGvuWoBKef8kN/szx7MAxk2NjpHNHIzpplJ8kBmzX76WXpAwz8w1oYvNHZnkTZX4Nv3PT/Jvf3329+e/IMnAGgmUv37z/MXL/461VdRf41rJJXLo3CDwwEAKXdyaYO/SP6FeOrzrQBh7YyzLIG3IeaG6kR6Kx7k8QJ+XtfR4pEvciKw82MtY4RivoBu0I0cV1ARqylNF/UNVRq8avFX+tcQ0ziJmmeL8lhtlwtsBTbiH4C33DSmRLmHgdkpJADTOUtfRoNwm7cgUBNRcblQuAVmCEEGmJbKLDM26sCWYMvfQutlflbUwTNyD2ur6YdJlgfYXDGtz3KdxnKNNX+hTon0U31BJBHQ/KWlTKFo5PGhATPGftbXVCDfJ4wxshlGoM8IFykNOGYaM7twSHPoDXbstxSYv6025Fn1ylw/AZPOobmf1pui6AizZ+rCXLjvRk+ssA330rtwf9upXL1CEwCMhb1taNYPbRgk5jb5emOYjekHbwvrSVAvYuA93rVhAIfTx1ReKQdegNgacNngP46KfCQLoqXPVetQYkC/rigygBRuY5vriFp2lEvS5epiBgZKk0acab1n1t0MzFKr/bdWA9oLGKXSP/SUesrLyMlGQv4ku0ug/okTDWMBvlARqR3h7DQJJtVGdfxMg2hqkbAl2mjBYFEPd1Ogu1JC/iS6ZRlFY2HZLIvgOQCejslV0huTgb9IVvDyG5dd/DMuvFaFPRNXDCvAcI4+o2i12V+9k58Bcm2YPhuG2OFRDDu8T5F6tj8BeXA/kelyqNY5rT7qrYYWId7gEJYNy3uHuo6YvKWw3HqxkSJxsBjDle9mmF+hMA6xm8APWY2LAZWSnL4B3yT2tB2wBrYCzcdGYFmmKCr6tCti6vChAgboONhrorGhbUW+QW9H3nUgUuNMCTOiIISEr2OLdGvZcFQJLjN7ogbJy9CA0URiRj0t36VSTvaZe/ABLmFYA7TZQvA1jI+j2M0HUxOpCa+uoQFzT9GATzynMKF8BWwz5FoBhvGlUQD753hF8Gi0WW/HCFBijyVxFMqmMgCSktD7+yAY2EGET2bAMBVVxktUYZliyb2Am8lZ0eUmmPtAh0fOk9ni6+vrQofHiVMRd2+XFxdGoxGtDYRum1raJcvJILDo02HMAF1upSVaHE56Cl8oGVtHVcRVVEDtQTnbOMbAnt8EVmJNGkBwKdLv8ihQ5o/7Xsqist02i66QzkDzSDpWvnnwuIcpVnaPqxcEpSsDgZMGsbu/U+HCRFWBdshZZxBQWY0OttSy6sJiBnZrbvC3Xbyt3Rd/B4miugihgAUMBf8o8hVKLYwyuYz5AMVC0M9+L/b5omfeEQQfdCQywiG8/i5leBeVXADko7AtvSHWzsaSOAaFQ1jr8+lC/BX661SzvLLZlzBa9rWr81CibadNqpfKFFIJu97Ypl9RgqLbQK0pUCeQzVnEMhslEi5XcCtGqUy8TR8rMStiZLQn8HPtbSVqYlbmAReQAZtPDm8mp0oxi+x/PwhQ0KD/014kv2Q2kYyJd1mTAFD/LoKPDzrTqbbQx9WoyThrXVMJMGqJyZ+NbpxbL8LpxeS0LsZTK1slhOed9sW8rgQwVrIqWgWE29ZdqDugFJlqhuSI/ZOK2WMYSx9ibTI69BaCsSLb+HWCBta5L5JhwA/vbQQyx02w2NLkqSDgkHD2YEFKoZLhRAMboMWFkEf/eDxst51VWiWdzjwaI3VA+TvIo23tf1OUWqAHg39uJjmUWQTyP4mtRbc6aw0Ckj/oWmJF7U2MaOuhmrLsRsMagN1AB5ZqxUiTpg5zfi+4G85FqnMKiklYZMjqOEc0dFzgyw/wIm/DKygqW4mr+CDZkMED9HopKg0BxpNo4taQFKM1ndEzLGYIZoLwEp6riVVmZHNacgUl4Ly9XvMWhh7UHUlbpfa2u55FkFFnz3p29B1kz1zV57oifQ5ToSpkL2BUoLg8flT0OlKm6KOgUzEwLIlniCkVlYop6fb0vOkw9Ms+50gfaGstcMmeRxwemoRE0FpSiJDefzpGz26q526OFZ6oeJaWplXmwXQq5ZZm1zkZkUob60pH8jnB3J8ajvdsRS82aJKI01iPXdo8mLfLpRCqdHql8vOjZrXZXTplKCTSTXAzr61wFEPXuy0yeKYz19kvjqO2F26LbH1pt3uiG8q21A6xNsxdFD+1xev1GrCjUEoxYhizYsBcY1dH9g/3/Jb1wOqC3x3cF3xoux8BEh7EbtSfoo0JtDW+E7JrcvuId1KnusDvs/Ax7P8dunK61Stl2mMe1iFxFOrcjYfvw3Np5aljcniN1ipAQYkAFGTMVlbMf5PetOOb5mjLWUFhQNZNjCQsABEaf72GZK0D0PMBeHaQnuRhXmfrPGOTYBIPTlJc6ey1faOOhFQUYpLKCLJmBgd50dzntTtIZlAUJxGnNqSRxkQbvglJ9Z7Cp3EqlIJz4BBvRSfU/aAI0vU7FbNoAP2HqBOXnnDr4fa/dJImL5dmIROgkhK4uAgNjULD7M3fEgYaaa7xmbAJDvVl+QU+Z/plNc0+Io7PRxkD1Sm5wss0P+0QjiBEvRENzLFhr9VBSQEdFD2TCZBb9No+ICXBXbeslplgLTeXsAO2gJS+KVj3RshT4CxOP6l0ynukZpeyKRCfsSgw//+LTTz93NBg3M4FAQwM6nlI0xvJ5riUpKcagIJ47Au6eG1JMUcyZlHYMufaQD9cYydEWo2HsAhZUL0M2OFZdK4suUtfqJMIg4Hx/hei4ZORVh9yyLo24/fICppIcTIQfvGQz5Am3Ly9A28Xt3748vcnfvkzvRwggi53Wu5GlJ3Zt6o/7JbY9oVfN3qf1aWv7PVboAV2DeWR7k4bwke6AX8NcYoBZZI4DA2wmgQEjNYfddXsY9BZEAqRV7jAqxj643XDuCgQOVIscyYJGHpkF6IxG2WH5jehwt4n7nN3VLNe/89zZgUnLjFXWxzcmGpD+YdVB6kzUhMWHyKqjMTPFuGD+UPKqeb8v1tdlLdxtlaQELNH86g5sfbV4lTq/KTvaxUmxCxKXaqd262S83If9QaYuMk93OBwnLe2HQnTWl2VDczI5xGlnnrWJ/Kc6R9VQ8DxMVUmgYt8Od6HwHndzoMl76KV+hzYYPumTxHHtcP/BRIRw2j9ouv7DrkEFYazoKXrkhCFxtrgzL9VxrOJd2S8uTciQE2BEPM8baMpcj9hmMyvkhoGol0XjgKjrjRw5aAxgeL4Mewt1p0qJOmhmduqcuD7IGEEni9ruA3haeTd4NcbXFFGhiEzEHt19oOZyPOikzY62M4+U6g+zvNaPbaefrO/L2vCuLf/YUKHaPl+Vtcz0HM9V6gcL3ejyX1M/7nhjYkcOcBmd9U5Q2AZqq+i8sTyuYIxSuDQFiRvNo5ukhSLBa6f8BOvgKplnfLiro+5tHwlpWTon2NwRGB/F2s/KZvhLKBOH/E6FZ9hI2+4Gz2DFttP1NCLBatq3p+riLKtXbhzYa4wrVyPKNvpmARzD0SyNIwhK+Iwux0IJtwJVfu9OvjM8f86NNPeXC+qS30C3zpoD7t3Giydj6SR8CY5W0ayD+a4SN5oWZNAAQ7qNYaHtJXp8b4GkBduha/LmpovZJgiMqjox3JJSgpH5KRWqihk7sCQVQeJ64BTbJkyMKYIv1N9+F+iwUUUqhWOy030JAzC9nbjOHMuwk0SYIEDbceTaLowJANGjnx45Su0jA2+7U8eNKDvDfrRgcJ3ko7WSeUszlE8hsyLCCkvxm9W1uXRj4iHkXJ5Ef1ySBDuaObeHL//kVAidA2F1c7uZfVcMxQt0HWkd7ce63NSCU2NeqJ42M8qQDIW+FJVlAA7q6khZ27SJDmil4YCW0UZyn63DK/qEKgVg9B6eFxpNbLfIZe3mpYbTOJ36KpNTmR6hvp3qSxb+qQ77GrVFTudxjQPZZomegMsopdQa7nSge6NiRg5GrBF5MwKEizN3MCzHJURmPIwQ7hJd3eEGhIqeZ0WN4+d1v9WxHG/UmXJ4h7hQD0MeegZW326B1m4YV18Z4TC3WxaHGpyU7qIEilwCTJP5LGkFRhJjbZgB15s2qZGxtnSnQA/4Q4pSXqgSu/nyTt5rVkSpyicniyQCc2p8PwK0pOJc+Tl0IP32WoAAZrh8E8Euj8Z9zlEkXyn6zAL42R+UkabEq6FH0UOHdAWATLvBRHExSHsr2XRNy73ZHF+2qx+hPj2uGTQLUnIJkg92oYSVTiOQ9s2l3JXaqinsWS8NF1Cs/HQ2UKH1aT7gu2yqbNiAfnE+UMWPZQTZTIeq6df9GOJS1QuyBEeMeEJS4dzB2OMKF1v2y+ULjt0j2cPBnqZTUjs4n6yyO6EqOcDqOPXEM3Ox0/nRA47XRZvLw5PGycJimEoATy5/u5W47fNeUOrqhVv0UH4XYRDI7OKZ72EPjkuASU/OIzw6HsTQWejiNqcgaHl10Dm+fvQpfIzZhR3ICmvpWgwJW8nzUVhjDDp84Brs0rqXK3mEsXSNHDttzQeHGYsDniYKV8c/S3cfN7ozY4QBDFb6vcIwA0MKOBGOIKyRGXUcJpZTZQareI/ej89P7M9tDcsek2hoDbldSw5jtrWbLaDeqRU+Hv8knk/IAHHLZCLPxUp7cXivp5NxgNXWt00vEo9/UDTBGvw8PYWeKqlovkK7KzmZBhNoBm7n+LWGNSVz9kHTDqXM4fr5v56+ltmo8+h9AKP70FUFRhp+JvnoqneHjs4p/PfscpWSgiM2Rnl0GRQWRip+pubB4QhQPD45xjBwuxhYws7PcavxHlL/wd12DgOTp+XlaM8pUGywzUYseAVbgbej+0Z+pIAGTcN8dOI4IE4xboBB4qkjw55KEZvD2uZ9hpNB5Z5tOQ/0tgotOBwnohAU99H5OfD0sTRQjywjDRyS8d44VkctA3NcVO9F1QB9IG5t//jxG1nbbH9VvuPQRJ9siCmjYgtaRa4RNovxMS04Mad/WTh5ifwYDEbZehkM8QjDc2Odqk6GrBx7oNaH5rAGcxUfci0Ecxa9nFW7Yw5lLPLKSye3c6XSGzziBpsdhmYPy2CdE0+oS+QKRAmhxKM5Zck5j0uwfGSS5WMTLUn66C2DlkrOroEqPHbToCHl2FzmY9Dv+xFUZXqG9w0WOdo2GKl5zvH2Nw4cYfvD3Ta4+D1248CQl/sGBBLeNpiq7q7hqujBmKh5iDm8KTS5VA5PhpKq/GCm19H2EmMq6mrDRB9uCeCRqRW/8O5T4yIcT7QpB+wilq7kjC5Hg8V7Q97YxYXxmrbAG1OnSkS3PwzmDkasaJ1gR04KPqgbNgIPq1G80ro2yYlKO6cVTtZ7HMs8Wq7M5XVoJjucfG/jiep95vqysMXvZZt4/B/wd3GZ3HTlTt0u4s7r0mnI0vdDiXpenqGin2+yHe1AHXFk05Bo3EaBuWBM/c9iU2P6reWR+xGjmv7/ZPa0Bp5imKUhvgq0yXADX0dnEs1QeC+QDfDQBGiCu3G88duxMclXT1hCj88cjGQ1z1pDoagJARWXbnDIkghbwlKbXjfO2QVQ+WGoUOADzaLNptkCW1CQSMuOb6JLGRW6mF083Ol9GpD1BoFHyvoRjZXIZ+/Dkn/U0FUAMiLkRSfHHsCcuWElmOx4fenCmnPnlpsTZrGyrWTKmouuM4OoxcadYTdGv7nhNXsdqGRMmCs5Yn0VqHPPH5l+nif+HG+qoJspZuv+ht9TRtocJCO8TgL2JrFXLg9SLuJPZl9t47FtpEwiz8M/so7Qdi26siftzLy/JoYTk11lPMErWCkd8ABLYGJOxnEr5XAEtkfjWJeqY24K1pgDx3AcbvSgpWOcrP0SQIlsmTBG+nQwVFPRGkzCYQRwxxXCzu3Gp/IyHpr28gLPN103G+XWxeWVsDq8/5XczTI3wGU6BRQshV7U/UEa+g92+s0ieuKDyqd4NbQUbTPGweylYmEP+sncrDh51PxBpnbOVvkK+4hh+EBcXK5vG/82t0tLQ82/B+nUU/rhI/quC/3YYaJTbmUN3chqbpfnl7LC3r1Tt8+THj/bCLQR8IIi71q1fnRLq2Ov0sXaU5dQOxFcfJFNH9xXgeBs+jD/Hzmcb6OrmuMtMqGY84NhawZMdXFd4v1Ed+523T1fp6o4YW11VbPTzMVtfKWzCYlbE5HsrskrsJ1UBBmbj0b3S7sGp39cwpslc2d7Tz6N4B0Os+GdzphQZjV2uruaqaB3okzHshILvGvKdvYnXz8xiV20aYB0dNNSUw/o09JHzKBCxA5gOIli6nYDvFgt4N5hlzMAm8ME5PJmgy52ssAeAmKuVwgDAcqojmRWNY4Bb5XSedOZSZemPHMFT9b9i75YCLPMRoT7Baoouqnh6svDCfA5AaXggBqILlYAHQypV3KMGze9SaEb8dzpV4jATq2HfXBf7Doh5AV806vkjkR+5mR+H82J15lTIU/vI26SSL0vJGTqswigAy8zduD9gQ8tJH5ab2ZGpDeObFmPtoWupRNO9aJsO5NeGMypPkYThajdYEgRdGzzRhXwnloLCCPzEhA7bO5feZVRqpIkFxr5hrY8o5CoCKWa3PH2EneJl5nMNoRxYSEjBSbjybeWKHrr5Q1q4uS0LVcFzEIyZcbGd4A91j7yWx87TE18GsY4+MEQJrFZ49ltB6Iwx6tDk2Czx7LoUHR45zlN+8JhgU25K0GRfzHFxnjdpPfJDIc4DOtjlNmXddPJ9KhSXe3W7TCCbz21annK7A6dP+wdTwGGxOMpRlt+6Pn6cuuBCF1yyu8GsCZD8Pi9F962dfSR6EAG4+m5kOkMfQuJ2rgsbIwlDSN75CKDMWYn3Gbwh24k+EO3EuhNxhjNY94sutMTOJWdbZWeItzaySd5Qb6xM3RNvD6Z38yo7EbNZnTtg/6ukOtw0d8GslnFzseCEnd5uknUow8MGSATnx46Ck0mpXNhLWGF05sfEBgP+j4DXRs1oLLK2SeJ/n2dSk1DHRqP77+hM5Ol/+H9mZT3h8a3XpuuRp+MOs4JStjmAaW/1IWr6Ra5NQv4wRTT8khnLn0mmujvYZma+sXY9BlT0h0qmkz+iQxmXC294xqrUFvvBMaouVMehKDPFYx7poKVdwwBq+nTE04Tc6oi1IIdlfAbmaJgT+VET2Pc5LbNSEdohHFR9xQuzJAQ/rF28z459RS79+2G8JliI9DRq2Jk9Xwk6J02KKv1LUxz54QoqxUyDPCUtFFmji+ay/8JS4NXOXqDD6+YjSG71tWo+JjSDJ9Xemp1C4e2tJprBUC/VQonmqw+VlYO6/yIfIsfujHax23uayfsVLZ5o9VGqIHSKYGuXlxONAEJhMD/qdZEqCKX5i7oZ8/GUEEOE7bahH2mpKpXzRW6LlinaRD1kRTFLn9odtEPTaAzI0Gdbp7++uzszatnEdHh7NU/3oznfCQugwA0fY7C8MQi614JnImux2Ln9ZszaBtpkei2c+QldqNqM3EYbBCWibqr8lhXpe7qUfLQhWVLEJAvB73Jd0pXHywB2X0g+qYKfcuFEXBL99qQ1XJ8g8ZqGqi8seIUoP5FG6vgxSEPoUmXi5yGowR3HEED7gHs5HUQx3CTF5CcgJkCdQQvC+ooVt4VJObGEAZr+paS1ZH7R45CGl1Rsnrg8hF78Qgb49E7SlbTN48wGLwAV1ZZb0UnMFjm3stjW/CL3aYv6WENeMHKNQSUip2wAXRpwCPF3XJLVW+Vav8UB/tY75TbNmwd8EvxUIkD6oGzpcoRkUXGS+B4ihz3wMj4AHNqOJ7iq237k1y89tJNdInmBP20G17dE+bY0D/8TTdncHy9uzPMWVB9DWIABt1wMAXDvd/ABq+Ae6HFgLearjwrzRuvtV8fTxvvqLeFluMVoMok5qinU/eubg9VRaoq1knDTjP3agl16vqhy1o/4B5WtdhgFDlilhNmczoa5a7Skc1tpyCLwrFE7c10CzMWHfX4XX8JfDHxDWTmoTXrivFkxpmLRUoy67zO2IK13TvRBXYEPQCUT1Y2DspkkbvbfiA4v/Td4I7nN7BRcQXTiLKfKkKxO0MFRgIsJcks+n/5LMAH3/f/+Mg8fQgBUyGPfRJhLJG9C2ida+aPC2g9jZ+qrmEGJvIrWFM2OfSNW8Bth9+YBhro79fPfkIp3hZr7fimtxh4NjWedrsDbrVfU0myEf26K+kTbIs834CMylPeFK8+wo6oTRKfnQFeZ4BXbD9ypRIPrkXVLmCXhN/ucj5wixFY+uyZ/Dbcufwg2Ln6SNLx3uSHkM6G5ow+T1vQpCzMYL5tmkoU9SvCv6ie6rvbZW7xQn239gj8/onuwnw46tG98EQLdXOT7IzPkp27PWwWEv9rRPqz7FdFX66fSSatxI2oFrrk5U8vXmWRsT2Sol/jNiTto09kTfrGFP6SD3N4Al7oix380FRGXPQnkg1i5itu/sfwsMJMLxVMs7FMKSfXfFCKR2jUp8nInbO235CnlQuaiwMNfRlPaq7xd9e8c0waEQKnPpeFN1c+Ua+gqfdFMAuAxTX0qttw3TFagaPPZ3nDz4KD5gnF6jARRc/i702neHLISzSKDjXmp9BnDOVHgYfEYEmfnjGyC/8p8QPgOON5jiZAnOfIX3kez3WCDXLbxx/9H+65kXw=\", \"model.py\": \"eNqtPWuT2zaS31OV/8Aw5QppU7TGeVRWiVLrOE7Kt7bjspP7cCodixIhiWuKVPiYR2bnv1934w2CmsnuzW48Egk0Go1+o4EJw/B1uT/0v/z4Jvgp7/OO9cG2qbu+HbZ92dRJsIFnVVmz4JS3+ZH1rO2SAH635TYJ8roItnlVbfLtxy4Nw/DTTz79ZNc2xyDLdkM/tCzLgvJ4atoe2tZNnyPQDluJp9v+5sQ69XW/VR//2TW1+lI1+31Z79X3Y94f1JeWqY/djYbVl0cmsNk2VcVoQp1E59e2YC0rfiq3vWhUwPy3Vd51TDVSj3QThmCN9/Q9ocH+bGo54Anwq8qNbPeO0KU3MF+YiHzxvL5JghdAwXxTAZQ3+QnfJsEH9sfA6i0zKFUPx9NNkHdBfVLPTrAA8AT+fyoE/O5jxfK2TvkSqZnsLrJu27RMLdD2wLYfT01Z97LJNq+buoTlzA55d8CGn37y+tdffnn5PljKBUj3rH8NH1kbZVkN/JBl8aeffHj3+tVv2dvnb15+gKZR2Ld5WYdJEF7mVVnQmuO3nnV9CM1fvf3t5fu3z19nL359/fubt7xP1uXHU8WyXQn/lAW2l4/a5ko+ATqxCmHg/2hlgt9wMEDtXT507D3SretZEb0falyTl23btPHi008C+AEGfZ+XHSuCpq6Aljvg5iAPiqFF8pskaTkcIG1Qs6vgH/l+Dw2AMzqYi2T0Tz8p2C7IWpYXGXJrhKu+oMWOg9kPuLhi4KuyPxBPpM2J1RGsbFMAystw6Hezb8MYV/AAa1kx0QF/WgYCVJMcpFWTFxFvEeuhBXlZ1gsaZCC7u3If8V8LyU8rkOcE0VkTXm+BT8U45S6A6Yr2qxB4LNs0TdcDzYe6CKH9Z8vgYj430UIKBv+dVwMnbhT+KFWE0z04Dl0fbFjArvNtDwQHQLh2YmBoWamRgWerm6zrG0IYBj474ktsHcjWQdkFu6bdlEXBavwU9AemFJcxohysYJflloVrnFy4PQ3h2cF+M4DxKbVDDRwUvHj3uwd4edzkVQ6ym9F60WxoIJAuFgaAnWy5YzmpSADNlZPZ8GEUV4MFcjDSyQJyoCCLpWiADWE9vsARvjBwB81sLweIUgZaiTMWCmB335KMqQQwAnbJ2psA+gOb0aoQwKA7VWUvxyfD0oESkIMfm4JVGX8crnkjFMgS9HVGvASNbzUyIfEcsj8ak3ARhPtN0YeJ0aLZ/BMJcUlvj0PVl6Q8rDakNxFKCyIF7ebpfH5hNkD2hkaXrIO3X1qvjvl1VrBTf4A3M/sN0A/NSAa/ofMOGjybW8Pmx02RZ9UFH9L36hm9GoHd49r0TcapOeqNOG1AD8OAX39tvpB8t2tzznaL4MLuuslJ2z+oBfsDhzbfcfFSi4ECZr8GtQv4l7BmW2jwWzsw833ZZaxGhQwTy9sOYfycV53V5gCdmz0wSHYCts268k9sdvHsWxtLZOM/hhwswZ/AOtC+8IyHy4qveEskGa7vxTdmG/5KSAMDkyDX0gB2x38dyw68E7Alnc2lH9nNIrgN2fUJOJEhIvJjgvzZsfaSnnK+R0MbQZf4TkNAzQaPEtURhcoWjLTs2bGLYt0JpNuBiDpGQrAwh6Ya+XPCvtMKSHmFKL09sktwWTYVSFABs9Xg7lzFbyAVwsSAX8BJy65gpDCOg+VyqhU6A6LVeRMhbA5ouqDZBfYQXEla8JSxAtfXtBloHk0skFv6A9p80B7BHHD9fhmct4+mPkuN/mrIPDg1XYnqCW1KwF8DQcGFkZjg2ncMlhmdLlz3KMSv6BRJUZTfXfFWL0gNtTD15sifmRSEqWr4aA9gDKGZYWh60IFk9GhsBEFWqsM6QTqZ4Cb4xiLFrep/p0lRIyQGDqYxc2HZQZxCUtvZFcOgBefUgb/KQAmYz0CBDLWwi9SGu5GigWD2U8s6VqMl6cD5BYdRjZMiAqB3iHhiqrFmCNHxvHy8knZ5puyyjp+CHEyzGg7ERIBUMiJ9d8CN2dzH38CkVmsDJdn+M6QQWbcM/PWqQQMnzF3GELPw7iyjviADDLxTKJCuC2dBJzEywcder1L71eEa/COkLniu3EfslI95FjMNInX6e5zM0HCR2W7HDX9mst60g5yQEyvCwAVxNbrMBYSJupFA9R7323CvNEjUFhdnp/pGeSeBCs2FkgeCQyjMcphwf9UEAuTIj0JkI78zZTXl/j53hNbQD9fMQDWWnheFIbwLJ+zfjbiYYypyBz8ORgzDrWXB3yyQcBbZUDONXwrlBaS3Sc7fUvhnvatPKYTAbZvLJlL7oVqBlhU4CthSvOXUON+GBsmOnDPMsYA+a5N+fhz5O4qhFxCitDZaHXDyMTff6kg2e/W62b6qC3bNWkFDCjDBdSz7LIvA5O6SoLmq4XUQvs7/vHmXtxCn9j8LooVueIc/2CulTrDE9BtH1MBBr6DToOCXiACtCEE7FSku7s+ovIJ/4dcPrC1t/4BziB4oRfsVEaAEQlu0REe2RFfJitw9M9CBuhp0VpUfWYIxxCwvCmjTUbB+WUJc3kB4gQoZaA46S8DCFehVNmpERZsyieqeFWXLg/eEQ1ioLMxqpCZATzhMpNoiIxmuYzP0p6HPzrUGsuKawfLgL9HXt5QiUuPY3WNrw3eSLBQekHsG8tgBBJtQYezwikkQwMn86rbE/tBkRRoHv8XkqOAn4T/03drpZFECM0sgfZH10EXIR0LZ0fsOUJgCCCSsWB1NwiVjpFrYr+4l+asa4JFR5KAlFjNyqsif60BngElqR1SHgKLe9wecGOi0vCOdFq2E+9mvQhGEWwTWqwAMWWDAtYTO0Oebr0bwm90OVK2AD9ywBctVw39RtJqvSZFuh2M3HCMLnTh2AZVVswUgprKiLrEtbdBfqBTi5NL2mLjGwMlZyK1mF2sF5++ntjmxtr/RYLtDfmIaaD+AX4ck4rp5PIBcyDiZWlMLacrkITmFKsSPmdCH0hMw9aExIKk4lJYilVBQxCKbZcby9RSNQKTXcaXHBF8JU4aw5gkmsIdj3S3HM9ADOFxOYoVYpaIzsfYYgJern4MD0KKJFpwt9DNIVgEat2hYR5qIgjvSEG2zxeRovU8xX2lyt1gKjgq4LsCkETDbrmry/stn1gKQ2XCtkLYgC4oJgciUDnBSrEReCAZy1FlGqCKsUERcwoUDo4vYphbv6ZCCB2W4ZzCWSQIbn5M6iI93HkS6qtwyV5VAkxbQw4Qm/stO5I1Be4AKupV1keLk+DyOEN7tQUJceGfx7NjDJi5m4EACfudiObs4j9vVgbUQOqqH3wfzxGjzxJRW9dheJkSmvolsIDHYT+PJD1p/x7FfZ5PakmZS2EKeo4TnmE0G9Y1JAyJmGI8MMMLXGDpDsOOpv+GaQCmKyJJfn90xyKolw8KcixGHDnhIoaD144/TvslolyjywDIFsoMQg68J9Ysie0KTyjI+i6XUXMivHHrH8nZ7EPG1regvFmtjlYFJy4Itw5Yi9NhOeWmNiCYPwA51CUsWmeO5iyBI7HDgrsr7uqn/ZG0T2dgujVEcwoO5g2DThKQ+r9xh1sHMtramOrfBSoPBm2vLIw2+wGXEBbh2nnGXQsGidV45KK+nGQMty+lmyRXq/RrR4DvnjRkpyhAhEtjO10KfT7O/n9lVfwuzCQz06FrsOADHfN4zvjW2TiRUyJr7zTHr8h2zO0fnnHsyUyrC1PHNC1jynqHmp51fztPB8w8vXr0KuBebb7fshFnezY3KBHzRBf/14de3wDVtCWE0cDKPc8iIAGLojK91xlBo7AbkqqzzCuWHASMw3OGIJp3brhr2AKhlaTdsojZc/e/z2f/ksz/ns79l6ye0Dxsm5LdIwHGcwtfyFMGbGN1vmYQMDaiAXgpxFKuLaBfubrmFn39V3GW3OOJq8bdv1maamCsh8MWgY6z9cvw2SqC4/sovrKZJFjqHIvfEBHFbnuXkdA/tXAeOYQaqwka8b65+aZvh9CKHIF6v5IdjXlXommKw9fr970T5gm2bwohJ0bzssXOXSl38zo1bW3akCA33yWiDuqlnRdl9VKijn0WZQsqepEGAm25bxIZD3PCMmBxbdjuyY9PeAD+1mLN7DNg+1mSRzNqhjIKrkEPwslPU4HktSqLvBpglBR7glKtuuNjsGusQsB3f7hPRSd9AKMQhDfW2OSLuuPf+dji+uwlwnz9VNPRG6twTPBtnB7jJBQShjITy0cdhs9HMzddPRM963Wa0bpzOAQK54aGcyjzK5P0opvvroa6JJ0/IGU/cxtuhbRnuWN301HzuAisp4zh+oSlmVMGseDTFEyAYTlnJNYBitI1G0WXLBsrLZJsBw9uF0ddOb6jorkMu3h5Zf2gKY/EBsWMmwonswPJT5FtSrOSQcQUDnoJAFndPTvkeJgyM97qsh2uDoQ1eBuk6BlgcUgHjlr3Wn4aD19106QmcBZDkY0rOdIclHFFYIdww9poe/QxYxGmBKqLZ0uRwo5uKntIXP71+HeHE4tR4P9kRnPs9dUQlL0Bsaesz69fTvUDqsK0edgvy1U+2j+ZmbHCNJiiInvfwajP0XDqS4NcPVl2N/Pkcl2m2r8rNVq7AkVakaIisoCga3GkzRovT4DmAugLDhju5LryWYbqbgZRVoEBNBYklS5QI3MD8Zmy3oyonUYuW+tfnPOeJsbIc8cm40ow8yQtZe3VDDan4Kjf96/02FfVmppiceOh6ylMYLAfXQoxAG9huw1TiMtQDqEzztc8WpT6p8WZbCDjpi4wWAU1kFFtBOO4zyfSHLGei9GtGelBrWS3hC2uXG8u5TlKLRKqj6cqSLi2U9yvUUXpqThFtcJNQWFIpepQ87+Bog5FiWwEUVFm8l9dPlK+m1jUVGRxa3z/Mdp/b+zJUJsZ3VQO+0xjIsOLYALK8pg7LzeoCORdc1KE7mImbzwNyIKpxnzR4eQnqFngc2Jtxwy50WFCza8OtCLrGBEguRCVt7TEHqlxDD0yXY868yk+dBkICGHBpytG856YAYTEHcEkGgbdMveiXVwcw4zpcVJYKo3DXoDkLliUStI8PcEsiAuerH8UmE+ZvtpTg0pqejCKXsYFdBhe0KrIjxId71I+YKy/3QzN0nijHWH6slB3QtDNgJsFYmHDHlQQbUm8PLazm0OFC/oOxE1WwjuGhcUStRdVbXCqxZIvyyLBYV23ZU5Vbr2rb5Nr7oHF2yC+bElzBDpecSaiYF2/z/VF6kKIC8gBeIxj2zovbTiSsoU91M0NrM+HUjqNB7hEEamHGbTj7GEky2SnFUjLTAYk9K+GMYvUdt/b6KUY3J4szzopxBVn5Z2NLCW51WerLeO3VW37bY2odoY6oPhbtyB+p3DqDB0LfCotyzE9camhzS8x0JEdjJ6UXhDSH4tlspcQjzzrLd+ey1AlWA8q6l6WjRJzk9Q4jyupm5Fv81IjkW36JJTOI3eyqLFR8A9PGaIRrs/ekkJ9v0SAihcApdOGhIO3zdgMeiiwQb9o0+LWW5b79AczNFeh/7TwecX1/KX9UMZLpqmAOq+6FEwqChxWPzbA/CJQgeqNMGOCOGagZailu7Z20qUn+bQUeU0SlSnoDVTO/qP2iBCzaXVpCqjHC7aLpRJ4JQwmQV0IsNlaNDTtMekrJHmGymq/RANjIjDxiskla/lYLu/3aLyeu+PKY4ozYPqyjhZJKjNoEPpv93FFhL/K/zlXyZPzUGngJslokFpi1nYIfix8nGu8SmT1jnfVDdssw0ZdhWkFogvFsJiUStV5v60iuqGTkeV6JnXXR+LTPRrZPZKuRZTfDXGh14ZSzGTEwmnoqodPPHmE9Or5xswHSEewGCBJNecMffiIiLetd41kNT96Aq7xHxXfCGSPHc/kovdgFb8ofvxPKAfWA8fD9hw/qm1k865n41Fubhk+D6GL+7Kvg8ePgWezpYq7nfW05YdJ3PNyAEE7YHaJJnLbdeRDjyETygDpS0YkkTwZSxdPiojRnv+F7gqqGzy3dUDU4va+4y1si5Kvc2ODGJsXWFO+Ip8rUZbSIC28wBm2drUkriQi/ejliBPNJ5RdTH4zSYL4cyF+JWYySjf7gFHWIPWheZSB2nD09USeLnJRVkODzzLkfcF/GS7XX1BYD6AfxtG0ce0Fqhn/RFfK7Q6ISM8ciNdcrks/PuEOZINjK77hKCKn2rcpYGT+evdc2RLUWDXgPl5j4424zeZ2ph7gZHhuqpjZZSzIexq0usekz2k2OJ3WFUVg6gQ5Vj6giB4/PMblfsNPVUqiMeXq3KLt83zJQiHR8S2p2uRSL4Fbx211o+WUPqIBx6+YQ2xEIPUPUg2TV7fIImYqZhE0lnd6N2BH1ErlbZG3FBrPgYjw3HhTwUgPCh3KkXFVS1cE9WSK9RWttQ2NP3EHl1QvKb5L1K/G9m+Ria4/PyE58kyIeWqAB+is4gv2Oh6Kixfd8EmekWzs9njXiUDzCyRvS6ELTTbGyerI+A6Y5PRRK8CS48EHqmgEPQQhQx7KOeM2IHsHT6RL3SVD4HXuYytSh0ME69TeGwUVsaSEwE+Qft+brurKWd2Ev9hMOEl1Kjt+E3hUrPDPXYmFjYby5V7/abKXBTM1Bobvk+Hqlljcda5R7SnUnRe1zc/9YpCrVBh7PWba6xrxoBgzxwF5uSzzuiuvoJng+F/37Mq+sA+Ni1xoMmCwwOOTVJeUsYeCuxACZ2+Z0rNofViz1sIIppVU9zIfBAfb5zK7FP78jqLJ+0hDoLVDcq+2GE7piXaCThyKlG8aTpRJG8ceocMstiQC7xIXdV6SCsTk0+H4Uy8iuT5amnfF3xQAJP/7wEAtq1FBBuzGyD1CT437/D/pRUJZq4fwR84O1Fg5iq4O1JxqyTdI3XyU+ZXe6WTrHDoV3Y9Ud+EKE2D1pTuXc/CB8F/17xet+bTFRXf6fV5ZzXE1Lj47g+cpmAhDb7qEy9659H3U00FfG6nxl7n0BkU5uruQtBIYliPVDT60VoT5RaSUPYmb3h1eiJIZPyDrF6cuv3efnEro6vpEnNfE0Gg7/Bc77C12Po9dxJc2vtLsWFtr+GkcglMmzmlqsz2FrXt8MZVXI80Emn+MZRmJ2WKLgX4Llz9934D+cdDPe0pU1Xhg/Q2jO34s98FfUhMiIr+HpSEbMKyfwMACojKpSQH9YfpXOk++/lpt58tAY+Nh0IQdAtLMaQEmc3Xju8pRgXpc71qEcmHdQyN5P1cFH2dAqzbaKtqdBTNd2W8ek8GwmygyeLybfn7NprM8s03cU0ImBLHBiIHmaWR3ZbETcPwGDCCRaWbh6C3GsqenbGISc43l9qnZXMug5O0agHlgXaJ0gNWhK5zqx2pLzgnVThKel53qJ+IG3RijZJ4kXx7T5PvBQ00Ha4qk6OFvo2xzkyW5JzVG6Rt2rwQUNlCCflMzi8Q6k0JLg24u/PTPOrhrA7j1ILeCnHsCTxViOtXeqq4yShHvmMAEHJiTL3vWcpsb86xOcgvTA2ZIKGacojWk7u3TG9vhyYmx3suQzcJlbhSQEYAHBlsgKVXL4pJSE5iGd6VR6+D6/ooLDmdiT0fD4xnXZIcvCYN8Z0QeoIfcyq47r1tAB77rwIsXziNI/HRU4EmvR3lYuazsfFTOnHNBNzWtmTqaYIBkd2PEdefUcjUTLenv3gHOwRjvJR1ZTKkYmP9BoSfcLkDsHKtq4z2lh1/d3tGPDzQn6RujlrVfUcT1iqk68gD6e+SjNLTzWZMSLY307dkbUCGecYzGAU99Ok0XFSddSjTYNBeH0AJ58wuQ+hfsDzoQ+aMrRGU33DO+MgXpCypETbDcRyyMvGsno2DX8g6eZ1uKmgfjBvGCCSYcTnoKPHKuvDL4+sWGtmbQBEpS+icEEHksnTNx4Imy33lm1/IbYMCwKruEo31c3vvO4Iaqa61ZCuZP1jPz4mxpIxDm38sFd+HCCCufekRyy7LwKWMocv+xnDR+AU7TwxX/R/b+lbnfT6e6zvqO8aeDs7Qo8g+usz/jKKXkXwejaH2tO+iKq4F6KiPvnzNjJuHPggTCMq+vW1lZ43YGPyCoKaCMDrSfuIHHw2FzTkQY7Y/5+btotmjMaTRs1vJqImAkMFY38RA/JLdc1vpJMNDJ70aP0GXhcZVVhB5pE/F3AyS1LqLu+abEiBrQVeIhYu0YREMMDBaIu5v3zN67hO0cHp+mYJE4Dk8RPg4tsPp/L/zzGEzg2a3PMjVkXmDlunGxGMRT4bLTHFXvuqUAFs9+kwn2wLguQFkGyV8IFfim0mn5szm7psWKJWQjbs33T0q2Los1ytU6EhC35rySw0F/Kb4nlio2v1XjIXEw2dydkv2sZVQFu2dIiWOL3Jb3zfuB0zVPV989b3Khxym/wkkTnZjhr/HARTK1D6EEXmp9fPAWdMj3YfiKkFO/N7GHoIYUHgrfV2r4jztCvAMD6bt9MJlItVjokclWsWNbJRR6z2XL8aMwU3VJ+SFwfTvDbBCOpjiPyexbnHmmzaLP0UUpbtzEzGhe7LO27UuUlUZ4pGPe9uJ1sxo01V8sc2BFPbGW7i4xfxhR5byaSl8euVjoEkFcZ8RM95PRTcjohHbmWMRinO89y2I6VeQuQdTiAoVeHzh7wKUYUeLzUPOuTBOaVPsYlDSMkzMCibTY5WCWIZJlz5NwYZuT72y1l6Azqnvv/0VT2WGVhjFEBfzyZM94PcnGze8mj7wa9kmB2Eae/nRmKF2uCY6f9V8x0mzS/73iax/sPf6+Vk6wvuAw0AfnlGXjZ2AiZu+9cV4GDVABvp1G9CycrNsTY5LXTfsz+mF9H1vhJkF+X3dK8OYCuKcbzzOLGYjFsosEltuLgDAyQLhl6LsuQ5AaMPJU9FuUl7WIu5+O6s1BKWCh4M6IR40QUcFtqU7K+eTD0VY+nTAH6e7YlLPSp0Od00nVGe4P6wIA4nyQuBGZt2cCUPBcQW9cXSQH9Wmx38ARyVsrB9bGM+6870l/pCs32Rlx/Zd9kZVoYcelxVhaUczfe9LCiIG0KD7NUjnt6xoWqCyH9hu+jZpvJO91cCEaTQ9N8XBjajm4G8aIe8JOhNJw5kYLlmC/FWEUcsRE4iQOKpmnJr8X1chkWgWeCBoTeuLWkEG0RTpMDgJbH4SiBwYyGthuTBfd+M7HEx7IeeuY0mrzkS6wosIP45LzXK0nxvvzitHJXVcRK7mO3rtdabBRfkifr6ej86pgBZEJ5/OZMZ2QNPOxlP3Haj1cfb8obPfScx/UwAmXBvG8mSO6whtrc9r71HAoeM44isfetC8LDVAqA591ocfOuz0Bd7aATbnGl+DmjvAFrI29rbAbzOmIJgLwmP62bq0jelJ8O/TZOy67BQ7Z5r0+7TJ3PFFdSRkIRU3hJlzRkqAaMDW/1QW92ax3B1cL4qqhbp24ypP0s3IBazdcx3lEgv17A11CIJH/yzC1CoOpNeEMXLnjR1e3vbLWNJkIV+LD6UjtSjsgDLR+0JNju31kMo7qp5RelIssCRik52aq+XLFtFHv2tQPjNjWhlWIsRrMPd4oh/Jm6iQ3WXYiHjY1qG2VlDQ2sPRiVvEt0wu5WDGztcpuXr2LtiWI7mLl/LS3HAv0AOwjll1pLnIBzxKhO2iPUGjlcuArbbasWlOsO2cGWvOlO4JlAF4sz3MbubehjJT8CTxGjff3sQpKTp2F8Tdz0D6YdzkMx4s17QJnj8RtqzyAkrrCdRscLYYTMBBgxknQ4/XjIt34kJvqaGEwCUPwHdgJkpujE+pMGmTla3u1sWFbd275tXt7cTvXm4q9HLLTuWPp9i8RSg6NjbVoU8bSQp5aAi5ssE0BVAu/NCxskZcUfI8nwbnHc0xMQOlKqa+eMmz36QzTR2MEvwMnCFDdgyDCtKqHKpVsEt+5Ad6P7Q4TClLfm8Nmes81yRe8zyJbkm3Wm+BB02rHspf7z+E3u4T+yJup1pA9/n3OviEXUUL57Eqd8L2N4uxf9NSLBcbPz/tcPRnXhxDix92ZF4m90Nu/hbDOePbBiqMgUyl6PzjjA9nUt4M1VWLOGf0xjSfPmGOBcI2O1qHRTUdS9yktigAfAqTtuSBmgx9cR9gOKXYibqje0u8lPr4YIj4Oge+eikPoX9NxCll6H7VDzGienEH+kULQ36nHutdeRWIKRCEzHt6ShoKx8emttBwrisVvc/PJ6Ww2FGZA//fClOtEvixD433M64F3uVdVc4YaN/EskLkDFF190FO2rA/4EI99uBzAedF8VTXMmi8O4bHmE48Ee+X/ulTvFwi7n+8J4z/lCnpyxQiC+4PUpPbK8jlbkSnsM1dpyp831Xy0EMut17NkR31TN9iOvWOgPKejhKvJHuE8nZ+KBCvQrsRylyPqmJ10wntvjiVD6iY/zHwtEx0OJI8tKiSlRwTwa/kGWc/Hh4+DLb+ZzpQg9AeDj4Jv52QkKgGrExKCkS4anE9j6Fub8GVpeAITKbHYxR+ama9jzy70m5PJR+uWuM4lJJBSPHdSyi/mcDtOabz7SVqaa4fJR4Ttm6+XcxLOIiTto4iFjcm8dh6NEJ4vvJ/7m2S78IGwk18p0kZ36Q2eaDa245/8A2mru5g==\", \"train.py\": \"eNq9O2tv3DiS3wPkP/AEGCvNdisOsru36BkNkE08s1nkYSSzh1sYhqBusd1a69EjUo57fP7vV8U3Kbbt3N1eECTdZLFYVSzWk50kyS9j1fSE7yjZNre0Ji9PT5fjMPU1eXP+d/K+udrxn//ygawrRtump+Rrw3dkpGzqqnVLyWZHN9f7oek5y58/e/7sl13DSDfUE8zBIO15M/RV2x7IZug5bMVIP5Cu4vt24G2zJk23H0bOyDASHOJNfwWgNQVsSZIgyu04dKQstxOfRlqWagWp+n7gFaJnCKVHx6t9NTJqBv7Jht58aYerK9jAfB+Y+bhvK74dxs4MsN3Em9Z+PVhY3nR2g2lqakVkXXGKc5pE/V1N7yu+syyTc/iqZvhhj3yridf9YUE+VPu9INVs1E/d/kAqEODeUl31NYzA331tB5lP+XVLq7E3orRHpjd8Y0Y+VH11RccFqfjQNZsSpVfWsPGCbKp+6JtN1Za7imnC4aRpq9Gkz58R+POO01Gcy2e6GcYasMlxoWnA0nk1MfqZ/jpRxmmtJtdT09YlCAz0jDM12FWbcSi3L8uO8rHZqNGbqm1QsiVXCEtQrW1zBdMZMvn82ftPP/989pkU+rzzK8rfw0c6pmXZVx1oEUCev/77l7O35dl/vvulfPPp7RnA//sfJYKabmFtVcstFP4Uz29FGB/Jf4nDy8jyR1I3G34BYws8tsuVJFEuKHEBYEVYsTiTs+IKOSD5sKd9SntQeyC2SCa+Xf45yfBUd3C+LVVY8U+z9VayaQvXNt+A1LZDW6cZKQqS5HhsibPK0gTk4GSO3KUSe2bhaMtosIyPh2BEkCFP/FB1rT9Jbzd0z8k7MX82jnCtgQ0YjSAB2TJKPk893hEBmyb/eP3hvSJ1kloExubXqQGLQ84POPs9+duXTx9JT2ktbAm9hUMiNQUZ1iDDAwhOqCbseUQASHXOqi0t51IA+YJZIQ0DS8WrfkNTpVzioDOHC0n9f1TtpGnXCh7SPwDCbmKcrClYLTKs/0k3PFEbKu5qoOvOIk/244BQQluTBUnERTPf6O2ejiC0npfj0IohBvLA/2t602xgxMEFhqNcDwNDYDDsAkE1toeS8UEYGRxpunXVIsOlkIca3dJKWF0w/kANcOMhhntcgm1Xt2QcvjJLKljhqhPf1aXGj9b0SJIZEyjh4yv8V11Xvce9sgINgPV4cAx0itapFlleN9stHak9pcyeolr10IFtkw8KNY+f3DU9sBW5U6ju9ZEdM0CaCH2wILleYbRmRTFd1rRCKWvKV9rgW1sizMsWVJSDvfk49PpmdtVt08GR7oZpZCAVAaLQXBihXl4kHmByqejCQy/XYC7gWLumnzh9EEkE3KBCMXvE/FCQU1fiUgRIuxycGAYNoE2wUY3b+su/I6/+dHqan5JllMrvyJ9g0mwdIAv3nl1PxVQeQ62vZwswEAzBLdXQvhAV4/rwHE58aiQYmCY6QvRjDhvgBpbT/qYZATV4pTQ5f3d+9v7dx7Pyy9mXL+8+fSzfnr1+KwbOzj+9+WtiRT3D5nAbIQhYS8PhBQo8BSEu1IHPcILSifAmx3/SLPOVWcx0A1hIDAXA2/x+trPVdHNNRLQIABCt0VQrIXyO6PyCjFNfNrVwswsijYgIONSIMUhgSrrKneEQ+lG+wqgTAgG8O/BJ+2O0OywwsXKnZEVSQY08DjWYaUIy19455ATL3JnMo9tDEKE+QBSDyKJse4gl82WjQy/ECuy7mGcgC7J8mQFuhJOTWeZb3m3VtNMo7MMd2EIwhcmwZnS8oSg2/VH5ow0Xo/rjPYELhhZ0QVILqWcz2FWdSg4UdQx0CVRcA5J/KywicwE0OQ/b9DdOeMtgm83QQajUYLIiAi+OGQocLnCjERrLvplGcCcc+DXCu0jUoCO5Sz9YOEXbo5f+ULgCfTqpGjm5U5jukfxh4qypKTnN8zuJ8j4J/IuEttduBAWRXkl4e5bO6Bd3RF+YYDi4OAYZcdmya9Bc4PhsDyshi+KHRwy0I46qHcGuHEQ4iXEe5qfaN4N6SDKs0AKZmC2tVErtdxsMkoFHsBkrb9FWO4jyTudteT98TTNwGONWmMPfnfzjpDuplyd/Pflw8uV32X15h9lfjv/8AQB39PZi9efL+8TZVxl7EatBGlNhNJS2mFZfrbvyho5MiF6YMAzU+A45Z/YwoglGR7thPMChyFwvhz34BGZcjqe+OLy48sB3wKLaF+6rzntzfyb1DZ+cw6vUtHSMLNNT/rqQT1gYDrngKlN1oNVIXuqxsgwi2/3BAe/3xyBlouzyXR8FVax5XKoPPnub/eQBjcMGVAhUOZPlDDUOaewOIz1fNMOVSKYBBwgPciDEJA/TDKUKqPhlnKh/ILsDe/rynyrI63zKzV0qHaVLhNalzojvZAYOO4JnK9cHDAMltFS5XEw6/sNcgK9jgzHApNNoFVyrQADH62ZciSR54WbPsehAzkvv+ti8Exsob3YsdJDTbbWGtKWTOAPkwOdlgAVTMYbqzMTt1NMy+XkEKH79VdTCp31LL4Q4hFCCmgIIC669Eht5oY8yCYHy7hr+TUEWYHmY0CB0v0BJOVxLhbInYFJjNDZBGmOm82mPhjGNxlCKoBw5dnUmzFRXJNGlRETLYfDBXHWFBUkPoZ+5rohQbhciksnCrj0kIUk0FjOpbQwqkuKuiBBmBNV2rDSml7lH9boSme2TIOivMOvNCRsoPDrIHyfz09OXxwNT59vjwWdkNLZI25iW9qmn3dH4Vsw4yMV3FzByTQA8MnoMu76osEro7G/N3qdrEcOWZaGubdqKMbE5cubZAA/WceMA+zSvLkNreRoXiWtnL22k7VwzVS50LvoLecPU/RNVPc/ERVe49Rd3TVjSTS3qhdk/OwIrhCxx2vzGWaAjL4XGA3GcwWYAf70fKZql2kpOjyg34HsFYRWFCf3/MIdAYYOE2A0B/8WlMv7g1lGRMHuBpA+oli4fa8xC0qrcBSwOWwiHzCCrOrDqoFl9s6WMm3FP4eSomzOwYRo3WDXQAgIOcX+vHowpiATMBUOQSwXFVhlq/wQEfRz4T2hXdQbyWVcf9QZLJJ9UI2+2YKkwBdGFNHInNzEJiKw7MN70MnEJ9NCnU3ZTcjx/FEwqcS1cBA5aeQg5iIX2dTqHAWKHkZdGPKLCXpaIuSyzHFK6ob2BgCuX5+3ohbhSYrV7NRS+B7hRiyJXMeTMo20RQe0pmubxKJgpIyK0c5NQ5bcNlm7kUjiiVNcHRQ9nFWnrWFujSyzHozDhhRENtqPkEAR9XBtUNpsQusxEEA6x9jhWBx3SAKpWqaTfTpA9pOqaKiZ0X4HeQFqIpSPJobBoyLnpMkTm00DnJUsLf1QxFYx6fMXmJGvBDETW7braXBdt1a3riki7twGyriAgXulzyNmh35T6RqVS+IsA2vU12ayAiuZJVxRFe+VM/IeRo4WVTa/8qwwVAmkkP6GgtNxg4QujNaKsQuvviSNLGXyVEHrl/BZbrbXb6FX5NXo34ChZzDpAZdNvhyKIkuZsicjNqrQIsVLI7EGHdBs3/4hue19taFjR0xHrvE2HGHIvhgUjKQaN4xE2Gi92w4T5dKrrFvmF6VxcXiSzpQn6hThWf9OuulWVmBJLT7oQcWTrx1YFVe54IWW5hOWyi8+WuHypl+s6935gDW9uaJLNufY7CDEqLO9HAFSao7qvEPA3mzfycOx2Lb2hbXEFIQDnY6pgF2iUTBtC94OAEgENcRMkIXusMrgXBpPsihfJSVqxDRZqMkZOUrECnZD4Jj+s4BPoE4N7mTGtuFnMMukHAyqkwyZme7X2LuGDPc5Yb/MdthTb1iD9sfgD5AA//JHIZoTpQnkdTIlymPh+wpxo4NrdCfmrcVC6zDP+ADSz/o7sHXSF89mRqe03Fu6JOG3IS/cIrHVQt7DQK5yW36WzgL0KAdmrEID22FKpywHi6bGpqbAo5oZAmoxX37FLJR9K9ko0scW18s9XGl7Rc5J2WUUJpZyQ8lRFf88rukvkSOqBmdhMHczTrIdarKw2Gsw0ORdgaJPpLVjW9kCAC/sG561EKKJQWm12UmFeqD4LGnV0VoTt28a0l9cTNreBNP+BhYm4F8SzlKqwKou9mpEwJ7/0dS2vh6+9OAuv0ePJCKepK3wBHwFsthoW7GNolsMiM+A7tdNgf7hwvCJ098umbhAvtukbLl2dftCBewGQ7VXin8fyFZOpOGb0kYJXGJ8QnYHJk8pl3hR8lWm5Hns4ZbennnvZxWy5SpPVaCz3RpN3pEqaeeZXRDOYFKkOfTpPyLL8qh3WafKdynLCHOUp0ZLBpeyw/1pFaNNCq4EKDTBmfkh/HmpSBog1Ld94MupORUVn1RBfN/DRk5s1bCJraSumYDEmcx23Z0M+Ix9oQgAVOWHSk2B1vqUcb5Xh/QQfgmieYv2beQfnaEdJMymeH7nxiYq38TBAlf4iv6VK1yDsLL6R6cyLlNBIqR3yGU0QI2AbUbWFYyHTa8boiKDKP3+hYwPq8BuIScTCRNFL6oHKYE29YTTmGQJkK1HmnolR5+rGz1KMFqlMREsuspRyYUVQ/SZmFiaiM1bCrVNJUxLBYaL7Uh89HMEDOWOwt1UMY6YMvb6qs4sEnQ64bs/omOGo2h/hcG4gBb+ag0Tq4owx4e+PSyVKgX9l5E35ig8pVd/RIK/4sSsTkbnIak71zbEdR5S923/03efRc576Xj7DOtZddaIu9aDC93rYt7/X5GCK5SzYDcN16uX4ZpsHHemCSDqF37RPlJwL1g8QDUByPCMvGi4q4+qOOK+DnEtkIaKKUii5BabbSL2wH+M1gSJeG1DCKALhBFCOMygi/sEHjniJ4smuPezAF3JgMXdZEyvkf3G3HTB0sXx56Ubr+hhErjcfnl9m2R3S6MpqCxS6h2ZczfzsfUqebgoVe4/6AQfj/DmrcJA3zW865buiPa6kZdNvwPMBLmMvmVuDwj8PwaaRB6/RmtS/spYUnPkD5aNHS0iijPTO8mmf68sCEhEn7lnL79235jKKYDLiSiIyeOwUI0uO1Zri9aaYFusiq3ypDqo+e72ezjKMYm4DHrMzj99ar8dXyNd5niW5CNqAx/Jv/NXFCEIu3BTOzcIvEg2i4rmjqNBHFMH3ReTRoXkRWBx53eriP1IyKmblJ/ls7VgJKotIH+zB6Ap5Hqd6ZIh3nXqteN9ZPPktrbv7/D1p8Y3Paf1KhTYG6AIvtGqqSAq76UMdJOhunUz3ICWg+1RNLf3RC9DNXroRgXE64CttqTiVCwv5XxYtl/lxvqzkxjyk7xxD1yaquGDrNZQcUDWLADgoShTfkKQEmIRBwm1ZcRHdWEe2ttCipy6jqETyXEAEjFiwy2cXJpezWACvavBDlxR70bEcfmYEbQJZ2I8BzDWle/tUXUc7EYtpdKEwn+ahg3Ih8R/zzOugXqB9wr4nPb3l+sKSr03byh+TxSJtwLQQbXkdGJDfk5dzsx7+jkcbdZnpuMn+Yx7GvjQNlpo8Et91eRTFMsxodhkcu/2xiOQCZIevU4eJH0kuVyZOvQuouzc1j+LOpe0+mR2eXKleDfwPU9T/bXrqacQbU5iQvz40PzP0tAEksYIMDTBZBrxO8Lcluv+iJPdh+Tjq8X+a2Ppt4lPbThP9sxKr2vLB67yltjJvSZgw3Qbi9Xg1YZB3Lmaw/74ZGxEwFmVZDxvx8zm7NK/qGjcSa7D7pMp02N3fVlPLC1W4eyGskCoAPojBe46wxIK5RYYl2odXy5ZGsCyRo+wFyJ89sj2ALDFI/YY9j3Tc8LQOe1qIF9dPxyabG0unHLbkw1L8Sko+ICswkhjxJ0gTDV4/K5Tu+VutQCcp9cEprkuD9eUAut+d3TY8lQ7cxZApHKCp+ieU4qeGZYkYyzIxP05C/M+f/Td14Evx\", \"viz.py\": \"eNrlPGtv20iS3wPkP/RxsRgyQzOSEyeOsQrgOM5ucHnB8S3mTtARbbElcU2RHJJyrPH6v19V9YPNh2zZ8czu3HgGCtnsrq6ud3UX6TjO3wSPElGWrFqIMi53Crhfs4u4XPEk/oVXcZaWbJYV+Jx9iOeL6q9vPrIzXookTkXgOM7jR48fzYpsycJwtqpWhQhDFi/zrKgYT9OskjCwl2pd8ipPsiqJz7CxvgtWpXCdw/nc8aze86m5/EeZpTaUhbkpF6sqThQeOTwBcBqJL9SRnlTrPE7n+sFhuvbZEU8SfpYIn33kOT712Vfx80qkU9GLcpCv8YrxkuVJZTqkq2W+xsY0N205TyNowZ6RQsCCM82SrCg1Lh+y+aesWKpu5XkieJEGS1EV8dR04qupz/JCTOMSSBrCBSAfTlfFBaBfZFN5iWg/fnT0+cPnk/DL4Yfj09NjNmLu40cM/pw/DQYvd9/sOj5cvt3bOx4M6HIweHX88hldHh29fHX4ki6PX7x6hx302L0Xb54fv1ID8I8uXx4+02D23j47fPVGddiH/3AscPPD+0/H4dfT//5w/BVxcXawy478DfD3AHn+8fDkP49PZI8MW0v8+V/8eYs/F/jzBX9+wp+/4M9r/HmCow9/ev81fPf502n49f3/4JKHg8ePTt+ffjhutg5xop/Co/86+ftx+OXz+0+nOOMerAZgFFU849MKpeKMT8/hgRaQ8RgFyWdlVUx89ilLxURSOhIzFqLahCifLkrfAQmdx3Zeo5AdSPJ9i6sFyWaQ5SJ1QcKyCORt5Kyq2c6+46GgLEBkEqEG4F8hQKNSkvwgyXjkyh6eNfU0SytxWbnFKg2juLDmrlY54B3F02oMWPuIC6CexGXVatStcm3UTD9xWk0UMggdZprFc6CJtVw1K3vKHPnYwcu6d4C9kD0IZAGTZMV6IwQl7wRC9W2MX0oV3QYD4JlIQjWgAWSa8LIMU74UJQAa4wUZOLzwGdiulJWgbSJy9ei4EsvS9Xx2LtajhC/PIs6w7QAJ5OLVeDjxvImEP4tTnoTQWpDlgzmW/NJ1sStobFZEY8c8dCYezS0f4NRq2TAZMJevkmo0UHgrUXAt4TBU9utGBcBqsRZstQKD3RrA2FlmERAMewFSG/sFc1G5xN44As1TtA9wmGePahFBPbHFtoqrRLjoSA6ksBEC6lpOoG5asIjqIKCr2Sy+pC5AYschiYcbJa95kc0L9Gv4jMWzLlvQQAyYSEpgv8P+ycB4FiKt6i6jq9aYa0fCBm0rOACmYVcSkWuaRF5LoI7T4NvMucLFXuMQWipdyYXiZWuuESB3pRdxfUVTXjsW/cpqnYiQX8aliz8H6I6Cw0vgMZsXSLuzLEsAydNiJYg2aLIUcXBAUMXT8zDnBV9KCCPnLKsWwFRSnTL+RYyaJlXJIcordUFxJUggE+ElgpMjXc8yYNQUlNCjUKGA+3zP63u+4CBYGGBoRQV60krqznI2aHNxVWi0UkF0GElnwpN8wUeD4LktaNhJUgt0MBKXSn6WHCyzJtI7DhwjKjUN40GDhVc1Jg65b+eANfzsmCZgf2YJGPjGE29iKYdj8AYAlmtsDLfavQnSAhGWkmUPkWtiT5/2zOn1gbLxWPLiXOAqlOdtIKDa2pO3u+52IV6IYg1AB8FwtzsbChY8fBbstinyLY5A/g7YMHi2p55dW3yEUDKercm/grTrII28MjitKa/EHOyeshlT5cAPWMel/5M0oa0SsEYzxjJjIOs4IYq6nLh+RqZVDSGsaiRs8Sv5hQjBckJg7GoHMZfa+o5a/drBGvet2tCsygXJ+woDERgbBW95xd8V6LGUU7ttvSDk+I/uXl6EShlsDcBgDelCwQAR9sCgDJiiMbW8rWpUZk7Hqo0u2qE3oATLc3gMJENzW46kKgtQ7SrMzunWa4Dcun+O0XsezXxaH/FtZFB/ihYYCXodQD/H730QzeCBXon1AODVawDLCVYqTPg6W1WuV7cjq+Ffl/CI8nj0bDDw2dlZdgnEnkJ+NXIq28A1xiDaPV0JHR4Bu0dXzhFEPWhCgfuoQ8RQ5nzMIqvh2rNkJaiyEHB3NT0wpAOujwz/gQwQVlYhCDrkJiPnz8GLmUYPRXSaZJCUAYKqbT7FxCURU7NwrZeupj6ELkY0akVs9zYYYXctJp3uyvCOu5y1g+8o+5aWfAnBrvuEFwVfg31I8wCyL7yxQuG60WdBEGjpBms3J1lBsycBjAeT2gup538ZsXbi0I3UaSIXJuIlQXIveIIOGC0JXZLXpDn0BCn4HNItGLVKY7BrOB4sYpnzqXBBhBQCO2zod1AASYO0VoxgCPi1F8+9Juk24DNWs042IKZpi9lqSMkoBMIyvyybmYa/te3pNy2hb0JWvKbQSAeAfk8w3c53bHGHx7ZtdHUwrSny8youRAS9bE9eB+OYcUK0hbk1RbcOxFxxGi4hCo/DJJuDLlBGCuRqN1rw7EGiKCBKaAwxTaofnxZZOBuaTubeeEAyhTHgRZmPyk30YoIons1Ega7QlSoP+rlapqVXy68abAsrj8GV/x3Zfoz4uLNGvgVR2fS8ZJrvO8R34IVIIlCtKwXv2mRUkJTOZTZVT4H5jkQoTrKpDBcmzdSn6WdlQAFCWCA0d0hiL0F4VldYTw9Yi28T9h+jThfUnVY3CdIkbWBg+CUtAu1euTpD0S8RjWfkKyggdocvfbYX7GqEcp5CUFRvr+CfewexAeONDTs3NMjQ1N8E/yYJO5wC4/h03bomj9kHcKMoAlZ4vfOufd1AzsoPMFD3mSvhgkD6aGLkBZHMZ2sK/NEfQbRYecj3X+Lc5ZTASLLaaYSERHYKyT0MBsBRYvLYTEKBqoQnQ9X28xocYtMLTKG5AZR+2spIUFKkoDbk229grfKqkXOKjUC7J0/s1GRgy/jNUGvkDUzQ5DjSRqwJeOj16ZnPapsKpBfpaom3wlW67LUCXcKIX14gYLdO6xmlQSPnTy/oz+mmZCawHwGZDb4nooQJHUllVE/IxlUuTp6iRQpMD+V+gZKeGfgAUsjW/l7fwEua03XeZFlZ0favZfANoN5UtwFHSqyrBXfLkYmYizRyTe9XVod2Dm+HhatcLlit2/mgbDE7Ih/s3OYpvVuJRK5du03tOvK123k8VvaBaxMyMUrT7WSboEkXEsruLXDahkxDwfQL/b+dUJHZVoEAWKZWoAJEqgF3Ykvl8DA+7k9a7F3IOgFoxNUWDB1ay0TqhrDahMD24K2i4CdEA9/GfNIO1YqwhPQhWoHM/O7CtPG4GYwZdqJlciZNVx13XLVx0vvBXsNL32KkZVtrMmOqjOJR+0az/RDmpm1qOnNvObzfcuycABD2VUnHAxiQDeZLSeuNelpLqaMY0BD6hkgbjGBByweWamsH/bcX72auUS+zFAA8omAQiDQ9zzOIpU2rVgM1GrUAl+XixmWAp4fnYl1qQbfjXbBZ1NHTIqtOICEbUt3usJtpRxJqdDOO0Oh519tq7XCISvuioVBnvLhJZ7s0m+iQZCxXN6ZAqZXl9hAHxoloLlQ4c4b5TyN4GQS7xiL0aLZJhAy7iuxbqf3qWE3Yx032mg0mtv5OeQVg3Rao1vo7T/tA4x437riO6PRUR2rm4LYc7e6ZNR2Z8ewMkqZzvSLQ4YLPUaTJobmbCR8sBU9BuNSGiUrcZEw3CAbaNiwhSJCJLJ6KQUSIhzA7HeOjZJzPhJ7BoGAfWikWOpDaAbh4uVqGi2xVlHiu9oQ9eyFh3zQMVDMPzwTIhwiXcbqqhBr8YtDUszAVIqLNA6w8CKYiTtx6OU8MpZ42sCZy6Ec8jZpLeq2D3sGv6ES+GnNyL/fx3lg9RN+Sk1Owxrd6EWhwIB+IgWZAu5pemqqjqxZ9r53bHY8dWNOWKURjPJ0uQL5diCdBU4cABMR45KzyXBQgj7PKXv/+d3uwplO63YnJM9gI7G8Rn63oGOyhwzP43z7pvbcPW/I0ngHPtjlll7uuoR7SOGdXFtDaErpyyjyJK+eA0b/o3xBhuJdn7wB5leJj3DvSMMdqEOiqXB51AiUdU/uETqTleAjGrpu5LnVBk6/2ONTWRp0rO6DvleOZAgDsa1GxsUfU69xxlWrBl3ILl8v9K7SBFiTt1sidQEdwKHvbeka0lEO5DdwACVZnELzcBVk3XtN2y3rttVO+hQp20m92R2S+gxtp2m1KdmDKbthIfMMII6RCKLedBaMfv2Q/MtdsxiHyRAuf6b0MdSs9EoEOpA3ytPfqCU+2cNx7HcsIThZNW5LNnU74jufYpXvZUKZNMf4Rie/dLTJpTclcQIARLt59rTNhwN5aduUBAvsbNy7ubzO7NvBGuxmq6kE0LzDislPmIJvtox6fpZhwJ4CvKoBoH/OSpEofDmODcsFzMdYBGJ4B0/PXbLjXOdpR4QDAhvEvG52VF3+l7MSiEOUiSyITsiCGPEU1lvNClPSU7cpzdYkHQm2GSlQWlH2rN8RpNq91OC03+2/oZHTZLHoMUH01cNLsiN6A6lquaMhB8Hwm61pqskossSgIjbQ8TPJ69grJschJsDYRZsQWyLL4yJkK8Du4Q31h3xhJwx8Tr35bgOASDnIVry3yyjIbqfMdhwtOakXhPa162nNspXs0Jegh/PB3emASLYWcFFE80iCPIs2RdAXKr9x0rOMcaTiK+UyCk4dGJaNq082lcgX/1vZ3NWK8xHNH15w76g0wtXh1DDWyDSiVQMIAURQlajYkrRdxJEZOPAcJwzAqTskxmRZ7dbUQ1nv3feiQzgFCoGMWGVdLWeU0xDI+kUfxsrRrB/qAS7UNpVdzOz2Qzyk4GPDMeVbG6Uxep2Kurjug28TsAXkrDZUBQmNC+1y1JFA4sLfvbRtWSD0jeDhyf98cGS5lykeKDHRaZN/6UJ2CyEA2jDeoy5A9jQb4L79EIvMyF9Nq5PBVldkFD6TXGA/QLIjiyHIjvUbfZz2zW7xrO+86AFMqojSy44/v1l17/S+FwM0WYOf0fv4fUb/r2KbbP8m+7XyqxarW8o9EtV9hc0+f9akyEJFSRBndsiHfssKXYc1Ix+ZqbXb9ujZpg3rmtHvQU4ekcxN7Qqr16YKIZncBEc0UCPmaAR2TzOJ6h/02CNaqVe1Rcznew0GnAqbmStvHDs25O71vqt4xbA/AaWF0eBsw8srfaY2WvDwnQQNzvOSBvA3xnZVQ/LziSe2SMAPcbMHkQG22LuIihlBUUWuk3r9wyYoNlRWjvIuq4C23h42eh0biQS2ctQhb0f6Apg1ijrY9Yy7wh33VmdJvZNu0kN/JwEHIhHXx/NsNJq2OrkLQFV5uPn7s2KL6HBJG61NIA0iHD5bhNGpfd9rurNGQwa8RbZ82mneOOsH1OqwAh2ZkvQ7zIjv7t4u2cQfnoPetGNy9UhkSFp1LE2TX5YFx2B0MhsZQYb0MBJqy4y+iyMowic+Fi6MbnXQaOtj6cGSAsd7L2iama1WvqGK/1+xFnTfKTFvtA+ltrXobSA6xImvJrfAMSEZv59j1gvKZh9s9FlivGf7vNwu0GvACMKkujW81oyFtZarAwWav5uNu9akpxAbWxulK2GtCaQunalPOrAZbxwe+vZiJVUNJycOzXbtmJgd5qOhHZ56ksVr23QbCvpnYAlFOIZdBd7Sauhqc9ThCBaN2uoKHKK1WVWvPmPpMuzta76Hp8mXmXhEGlM97nTPshrSgZNkJPWqHNoRX7f1amBVu5B7HJZqVympYU4MmmWmuvGuSUZgLiFXpwq8+Ktt41Mr1IzE0xl2DnDTLZz30sfUMRgwtardlrGaZ38tsv4dBNbiuTLa1CRNeDFSCbIUne13d8lvbxspXd3QsKMARJfU0mwW8R6Cbg79TqJcxcWOzWN9VpLcT54847Y4+TnOvLDSMbHfOOTdXpO1aHqAr487Sns357cT9AQW0K5y052hU46CtMJJRtbI9tXt3tNHiv1REA6TXUrX6WHzlLb7yzXztrTQMevl6E2/5d/FW7gzVnNVsbVOhI91jjBrA3+h/9aqe019zVQet8sm6AuGEp1G2ZH9dQRLkbIrlKaZlX7IyruILwU7uVELUiOfvCaMZ139Oxc5FuXOCJ4snn4+2rWPEA2R64V7ajXJ0hRZSGc4D/3dyXFwHyliT2Dk/3HxunBd/kAD7/1sIDGlXK94FfH8X8a75wITP5BcmWjFC/wcotoiCyTVpkHRjQLWCA93JPN8c+PbD05ZSxql3CHdzHkUiChsLRnKCXQDrYbtc6QK2jZQlmk0PY7C9ZwzRoZLfRd9e241xxY1c7wHs38TO33t0/KuowPeL/51Evxks3y8wvqc2bBtP/6uV4rdWiK4ydIK2E0mSO8dpXwzZHiA8M8B2JDp/rFjNhFx3DdVEocr81OZqO2QzHVpv8D9IjPYdRetZEQnM+gx+Ab5jqg5a3bFTAqWhwTH1ehM8+5gCi/DzRWP5Pj7+TsCMChQtcq5gpLLc3oped+rjaGatjQuBL7zL0rjndykax43EF/IgXAKkA6Tn+55dFJfNZoCZPlJXRlIZRSoQdHckApgL0muO9ILjABiOO72WXcJvUFk3KI+uGVpo/a01ebdRonHGqeReV8Ut3DX70SBH6I9VaY7PNFRp0G2MW5Vw1HaHSjhNFEDAr8vi0Xwiei08el8FJDHCjV1pjNVL7j9i+dJg38e3upAuAI0+DgOBLzzTd3JZLtVAocm4smHImqNNpUH7G865wHKPARBo6PgHktEfJtcQwo+u6KtL2KqE+IcJGiLPoeWHvq6wogXjd54KVHLXm3Ttc7ykHf9gsLfJdn+dyhqWO5puWcLnKgTvXQn4RRQ7EtS7IVDWiChcG1H8Ny0MrC0naR4xY7OdnQlOXxmUH8Xj+OL75i+obP5iSvvbKiR+oZR5u11y126547dW6C1GG3Cn169k16ssN/W7tkW3V2pbcnkIGiwEj9xnA2+rMZZ53zaDt8w1YCiNNUTg2lgTuajwOMvHNu3onWy7gV5IaDTguz2waPMecfNNogXON3aU/NBrRHBvrwyaLgHgSOLQb2sH29hZlN6cFr27wVpcbvUaccdYvFPI389IoMyDi84TPL90QnRTzPHqKut/2RuBMkveXAusD6Px4ziFWIJfAMRgISChVpgldetB3gvMMoxEsKm9maXOx2uf+KT/GykAwVqK3+lvv6h7S9/2G5C93Sc91JIMVOU8bStpUauRmeOUjT3ERkK94Wl/Ma3KIjfEvbpUR5tx+Z429FJfYpUbhfaQyd0t8F35a1e2/aZM7itYlkPqyqDNo+1KDBplMdO3eXcTEGuz+f4wulkQwTLtN9Kg5y2pB9INOobPBX2gzP76TzeaCOf4iswBc525elfmNANzw9QdGQ3/Zgj6NSdXviwjT88Q0ld6J0ffdkABkZYr+ZVECxwBwrcrw0iA9eOlsL/CwuzGsorqTg3QWPfdEzrVsPlZGWIngg0NDBrAKeIr4387/EIbztZrGzbq13WWJe24jBZ8pn0cuVP6xAtxQH9P1corOjVXmyI9yd+OzaCvuE60H2lNX9dhSUSaltxM/fjR/wHHOUgc\"}")
expected_hashes = json.loads("{\"checkpoint.py\": \"8842aac9d9cb53226d9aac743f44dfa2a54dbe374864685c323b5ddef905f8f1\", \"config/data.json\": \"b5954881e80826bedbe1ca9eb3fdc5fbeeef609d0f5c8edbe9eef297d2f1d7ba\", \"config/data.smoke.json\": \"9ce6ffa8e094991f310dce0c7dd08d8fd63d8e873bbd059230e7c85c53854e5f\", \"config/orchestration.json\": \"6590d0a2d067e7ce8a75b2ef489e5882fe2007d4c150ebb535d53b61ee7e8118\", \"config/report.json\": \"847057bd82f6113f47ac60da728b970fbd8b97b6bf9a8015a087992568e66bc3\", \"config/train.json\": \"ea206d6c70e49d896b3046cf43d7facd02e02b0118b27665897b7a0197a9478d\", \"config/train.smoke.json\": \"c25186d3189c223e370df3043834741fa12b05f6e8e065e65d4a4803cb693e40\", \"data.py\": \"192070225e90537458721e9c13d198b604bb9308ee66692f6223be942fb8a45e\", \"make_report.py\": \"746537d648ac74aac26d7ef9e04291cd5034bb202c1cdb4c43f3139d89209f95\", \"model.py\": \"335fdd99f24f16b79eaf170cfcbaf051d576a5b4c2e2207ba50cb9eb27106996\", \"train.py\": \"9f3f29ad50b5edb6cbabf683f4dec29f6046a0931238a45b45895998a0c6fe6f\", \"viz.py\": \"2f6feda08ffe0d1abf476c291a33e6d86dd0761f021de440967628da0f64714e\"}")
for relative, encoded_content in encoded_files.items():
    destination = SOURCE_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    payload = zlib.decompress(base64.b64decode(encoded_content))
    payload.decode("utf-8")
    observed_hash = hashlib.sha256(payload).hexdigest()
    if observed_hash != expected_hashes[relative]:
        raise RuntimeError(f"Embedded source integrity failure: {relative}")
    destination.write_bytes(payload)
print(f"Extracted {len(encoded_files)} versioned source/config files")


In [ ]:
PRESIGNED_CONFIG_ZLIB_B64 = ''
if PRESIGNED_CONFIG_ZLIB_B64:
    presigned_path = PROJECT_DIR / "s3_presigned_config.json"
    presigned_path.write_bytes(zlib.decompress(base64.b64decode(PRESIGNED_CONFIG_ZLIB_B64)))
    presigned = json.loads(presigned_path.read_text(encoding="utf-8"))
    os.environ["S3_PRESIGNED_CONFIG_PATH"] = str(presigned_path)
    os.environ["S3_BUCKET"] = presigned["bucket"]
    os.environ["S3_PREFIX"] = presigned["s3_prefix"]
    os.environ["RUN_ID"] = presigned["run_id"]
    os.environ["AWS_REGION"] = presigned["aws_region"]
    os.environ["AWS_DEFAULT_REGION"] = presigned["aws_region"]
    print("Loaded short-lived object-scoped S3 operations; no AWS key is embedded.")
else:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    aliases = {
        "AWS_ACCESS_KEY_ID": ("AWS_ACCESS_KEY_ID",),
        "AWS_SECRET_ACCESS_KEY": ("AWS_SECRET_ACCESS_KEY",),
        "AWS_REGION": ("AWS_REGION", "AWS_DEFAULT_REGION"),
        "AWS_DEFAULT_REGION": ("AWS_DEFAULT_REGION", "AWS_REGION"),
        "S3_BUCKET": ("S3_BUCKET",),
        "S3_PREFIX": ("S3_PREFIX",),
    }
    missing = []
    for environment_name, candidates in aliases.items():
        value = None
        for candidate in candidates:
            try:
                value = client.get_secret(candidate)
            except Exception:
                value = None
            if value:
                break
        if value:
            os.environ[environment_name] = value
        else:
            missing.append("/".join(candidates))
    if missing:
        raise RuntimeError("Missing S3 configuration: " + ", ".join(sorted(set(missing))))
os.environ["PYTHONHASHSEED"] = "2026"
print("S3 environment configured; credential values were not printed.")


In [ ]:
required = {
    "lightgbm": "lightgbm>=4.0,<5",
    "boto3": "boto3>=1.34,<2",
    "requests": "requests>=2.31,<3",
}
missing = []
for module, requirement in required.items():
    try:
        imported = __import__(module)
        if module == "lightgbm" and not str(imported.__version__).startswith("4."):
            missing.append(requirement)
    except ImportError:
        missing.append(requirement)
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
import lightgbm as lgb
import psutil
host_ram_gib = psutil.virtual_memory().total / (1024 ** 3)
print(f"LightGBM={lgb.__version__}; LightGBM device=CPU; Kaggle accelerator=none; RAM={host_ram_gib:.1f} GiB")


In [ ]:
preferred = Path("/kaggle/input/cicddos2019-parquet")
if preferred.exists():
    data_dir = preferred
else:
    parquet_files = sorted(Path("/kaggle/input").rglob("*.parquet"))
    if not parquet_files:
        raise FileNotFoundError("No Parquet files found in attached Kaggle inputs")
    data_dir = Path(os.path.commonpath([str(path.parent) for path in parquet_files]))
print(f"Preparing deterministic leakage-safe splits from {data_dir}")
data_command = [
    sys.executable, str(SOURCE_DIR / "data.py"),
    "--config", str(SOURCE_DIR / "config/data.smoke.json"),
    "--data-dir", str(data_dir),
    "--output-dir", str(PREPARED_DIR),
]
if os.environ.get("RUN_ID"):
    data_command.extend([
        "--s3-config", str(SOURCE_DIR / "config/train.smoke.json"),
        "--run-id", os.environ["RUN_ID"],
        "--maximum-hours", "12",
        "--stop-before-minutes", "30",
    ])
data_result = subprocess.run(data_command, cwd=SOURCE_DIR, check=False)
if data_result.returncode not in (0, 75):
    raise subprocess.CalledProcessError(data_result.returncode, data_command)
PREPROCESSING_PAUSED = data_result.returncode == 75
if PREPROCESSING_PAUSED:
    print("Preprocessing paused after a durable source-file checkpoint; training is deferred to the next session.")


In [ ]:
if PREPROCESSING_PAUSED:
    print("Skipping training in this session because preprocessing will resume first.")
else:
    train_command = [
    sys.executable, str(SOURCE_DIR / "train.py"),
    "--config", str(SOURCE_DIR / "config/train.smoke.json"),
    "--prepared-data-dir", str(PREPARED_DIR),
    "--output-dir", str(RUNS_DIR),
    "--upload-checkpoints-to-s3",
    ]

    if os.environ.get("RUN_ID"):
        train_command.extend(["--run-id", os.environ["RUN_ID"]])
    result = subprocess.run(train_command, cwd=SOURCE_DIR, check=False)
    if result.returncode not in (0, 75):
        raise subprocess.CalledProcessError(result.returncode, train_command)
    if result.returncode == 75:
        print("Session paused only after a verified checkpoint; the watchdog may launch the next session.")
    else:
        print("Training reached iteration 100 and final reporting completed or remains durably retryable.")


In [ ]:
active_path = RUNS_DIR / "active_run.json"
if active_path.exists():
    active = json.loads(active_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "run_id": active.get("run_id"),
        "status": active.get("status"),
        "current_iteration": active.get("current_iteration"),
    }, indent=2))
else:
    print("No active run pointer was created.")
